
# Análise dos cargos e remuneração dos servidores federais: Outubro de 2025

**Autor:** Rafael Theodoro Rocha

**Instituição:** PUC-Rio | Pós-Graduação em Data Science & Analytics

**Ano:** 2025

---

## ✅ Checklist do MVP (Requisitos do Projeto)

Este MVP de Engenharia de Dados segue a metodologia proposta na disciplina:

- [✅] **Objetivo do Trabalho:** Definição clara do problema e perguntas a responder.
- [✅] **Plataforma:** Utilização do **Databricks Free Edition**.
- [✅] **Fonte de Dados:** [portaldatransparencia.gov.br](portaldatransparencia.gov.br).
- [✅] **Coleta e Armazenamento:** Extração e armazenamento na nuvem.
- [✅] **Modelagem:** Construção de modelo de dados e Catálogo de Dados básico.
- [ ] **Carga:** Processo de ETL (Extração, Transformação e Carga) documentado.
- [ ] **Análise - Qualidade:** Análise da qualidade dos dados por atributo.
- [ ] **Análise - Solução:** Respostas às perguntas do objetivo, com discussão dos resultados.
- [ ] **Autoavaliação:** Discussão sobre objetivos atingidos, dificuldades e trabalhos futuros.

---


## 1. Objetivo do Trabalho

O objetivo principal deste MVP é realizar uma análise detalhada dos cargos e da remuneração dos servidores públicos federais civis do Poder Executivo, utilizando dados de Outubro de 2025 provenientes do Portal da Transparência. O escopo da análise exclui intencionalmente as carreiras militares e servidores do Banco Central (BACEN).

Para isso, será estruturado um pipeline completo em ambiente Databricks, abrangendo a ingestão dos múltiplos datasets, inspeção e limpeza inicial, tratamento e transformação dos dados. O processo culminará na criação de uma estrutura de dados (tabela tratada) que suporte a análise exploratória, o cálculo de métricas e a produção de visualizações, permitindo compreender a composição da força de trabalho e a dinâmica remuneratória no serviço público federal.

### Perguntas Principais:

1.  Qual o quantitativo total de servidores (ativos e inativos) no governo federal?
2.  Qual a distribuição percentual dos tipos de vínculo dos servidores na ativa (e.g., efetivos/concursados, comissionados, temporários)?
3.  Qual o quantitativo de servidores em situação de afastamento ou licença?
4.  Dos servidores afastados, quantos são por licença-saúde, licença para interesse/capacitação ou licença-prêmio?
5.  Qual o Órgão que possui o maior quantitativo de servidores?
6.  Qual a carreira/cargo com o maior quantitativo de servidores?
7.  Qual o Órgão com a maior remuneração média?
8.  Qual o cargo com a melhor remuneração média no governo federal?
9.  Quantas vagas existem em vacância no governo federal?
10. Quais os Órgãos que possuem o maior número absoluto de vagas em vacância?
11. Quais os cargos que possuem o maior número absoluto de vagas em vacância?
12. Qual a diferença na remuneração média entre servidores do sexo masculino e feminino?
13. Qual a idade média ou o tempo médio de serviço dos servidores públicos federais?


## 2. Plataforma

Este projeto foi desenvolvido utilizando a plataforma **Databricks Free Edition** para processamento e análise dos dados.


In [0]:
# Comando para verificar a versão do Spark no ambiente Databricks
print(f"Versão do Spark utilizada: {spark.version}")


Versão do Spark utilizada: 4.0.0


## 3. Fonte de Dados

Os dados utilizados são dados extraídos do Portal da Transparência do Governo federal, disponíveis no endereço:

[portaldatransparencia.gov.br](portaldatransparencia.gov.br)

Foram utilizados os arquivos `.zip` do mês de referência de Outubro/2025:

*   `Servidores_SIAPE`
*   `Aposentados_SIAPE`


## 4. Coleta e Armazenamento

Nesta seção, realizaremos a ingestão dos dados brutos provenientes do Portal da Transparência. Os arquivos `.zip` (`Servidores_SIAPE` e `Aposentados_SIAPE`) serão descompactados e os CSVs resultantes serão carregados e armazenados em um local temporário na nuvem (e.g., DBFS/S3/ADLS), preparando o terreno para as etapas de tratamento subsequentes, conforme a metodologia do MVP.


### 4.1) Preparação do catálogo e schemas

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS servidores;

In [0]:
%sql
USE CATALOG servidores

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS staging;

CREATE SCHEMA IF NOT EXISTS bronze;

CREATE SCHEMA IF NOT EXISTS silver;

CREATE SCHEMA IF NOT EXISTS gold; 



###  4.2) Criação do volume para armazenamento e coleta dos dados 

In [0]:
%sql
USE CATALOG servidores;
USE SCHEMA staging;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS dadosabertos 

In [0]:
dbutils.fs.cp(
    "https://portaldatransparencia.gov.br/download-de-dados/servidores/202510_Servidores_SIAPE",
    "dbfs:/Volumes/servidores/staging/dadosabertos"
)

dbutils.fs.cp(
    "https://portaldatransparencia.gov.br/download-de-dados/servidores/202510_Aposentados_SIAPE",
    "dbfs:/Volumes/servidores/staging/dadosabertos"
)

True

In [0]:
import zipfile

with zipfile.ZipFile(
    "/Volumes/servidores/staging/dadosabertos/202510_Aposentados_SIAPE", 
    "r"
) as zip_ref:
    zip_ref.extractall(
        "/Volumes/servidores/staging/dadosabertos/aposentados"
    )

with zipfile.ZipFile(
    "/Volumes/servidores/staging/dadosabertos/202510_Servidores_SIAPE", 
    "r"
) as zip_ref:
    zip_ref.extractall(
        "/Volumes/servidores/staging/dadosabertos/ativa"
    )

## 5. Modelagem

A etapa de modelagem visa estruturar os dados brutos de forma que facilitem a análise e a resposta às perguntas principais do MVP. Aqui será definido se a abordagem será um modelo *flat* ou um esquema estrela simplificado. Também será criado um **Catálogo de Dados**, descrevendo minimamente os atributos principais, domínios esperados, e a linhagem dos dados.


### 5.1) Camada bronze

A camada bronze foi responsável pela ingestão e armazenamento dos dados brutos extraídos do Portal da Transparência. Os arquivos CSV referentes aos servidores ativos e aposentados foram lidos e carregados em DataFrames Spark. Em seguida, os dados foram normalizados quanto aos nomes das colunas para garantir compatibilidade com o Delta Lake. Por fim, os DataFrames foram salvos como Delta Tables (`servidores_cadastro`, `servidores_remuneracao`, `aposentados_cadastro`, `aposentados_remuneracao`) na camada bronze, preservando a integridade e o histórico dos dados originais para futuras etapas de tratamento e análise.

In [0]:
%sql

USE CATALOG servidores;
USE SCHEMA bronze;



Os arquivos CSV extraídos dos arquivos ZIP foram lidos diretamente para DataFrames Spark, utilizando opções adequadas de cabeçalho, separador e encoding para garantir a correta formação e estruturação dos dados conforme o formato original.

In [0]:
df_servidores_cadastro = spark.read.option("header", True).option("sep", ";").option("encoding", "latin1").csv("dbfs:/Volumes/servidores/staging/dadosabertos/ativa/202510_Cadastro.csv")
display(df_servidores_cadastro.limit(5))

Id_SERVIDOR_PORTAL,NOME,CPF,MATRICULA,DESCRICAO_CARGO,CLASSE_CARGO,REFERENCIA_CARGO,PADRAO_CARGO,NIVEL_CARGO,SIGLA_FUNCAO,NIVEL_FUNCAO,FUNCAO,CODIGO_ATIVIDADE,ATIVIDADE,OPCAO_PARCIAL,COD_UORG_LOTACAO,UORG_LOTACAO,COD_ORG_LOTACAO,ORG_LOTACAO,COD_ORGSUP_LOTACAO,ORGSUP_LOTACAO,COD_UORG_EXERCICIO,UORG_EXERCICIO,COD_ORG_EXERCICIO,ORG_EXERCICIO,COD_ORGSUP_EXERCICIO,ORGSUP_EXERCICIO,COD_TIPO_VINCULO,TIPO_VINCULO,SITUACAO_VINCULO,DATA_INICIO_AFASTAMENTO,DATA_TERMINO_AFASTAMENTO,REGIME_JURIDICO,JORNADA_DE_TRABALHO,DATA_INGRESSO_CARGOFUNCAO,DATA_NOMEACAO_CARGOFUNCAO,DATA_INGRESSO_ORGAO,DOCUMENTO_INGRESSO_SERVICOPUBLICO,DATA_DIPLOMA_INGRESSO_SERVICOPUBLICO,DIPLOMA_INGRESSO_CARGOFUNCAO,DIPLOMA_INGRESSO_ORGAO,DIPLOMA_INGRESSO_SERVICOPUBLICO,UF_EXERCICIO
3174964,AARAO CARLOS LUZ MACAMBIRA,***.017.623-**,167****,BIBLIOTECARIO-DOCUMENTALISTA,E,00,018,000,-1,-1,Sem informação,-1,Sem informaç,null,26405000000040,DIRETORIA GERAL/CAMPUS SOBRAL,26405,Instituto Federal do Ceará,15000,Ministério da Educação,26405000000717,DIRETORIA DE ENSINO-SOB,26405,Instituto Federal do Ceará,15000,Ministério da Educação,2,Cargo,ATIVO PERMANENTE,null,null,REGIME JURIDICO UNICO,40 HORAS SEMANAIS,11/02/2009,null,29/12/2008,699,11/02/2009,null,LEI,PORTARIA,CE
2903139,AARAO FERREIRA LIMA NETO,***.116.132-**,143****,Sem informaç,null,-1,-1,-1,CD,0003,CARGO DE DIRECAO - CD - IFES,0099,DIRETOR(A),S,26239000001515,NUCLEO DE DESENV AMAZONICO EM ENGENHARIA,26239,Universidade Federal do Pará,15000,Ministério da Educação,26239000000966,CENTRO DE PROCESSOS SELETIVOS,26239,Universidade Federal do Pará,15000,Ministério da Educação,1,Função,ATIVO PERMANENTE,null,null,REGIME JURIDICO UNICO,DEDICACAO EXCLUSIVA,23/11/2024,null,24/07/2006,2475,16/08/2006,null,PORTARIA,PORTARIA,PA
2903139,AARAO FERREIRA LIMA NETO,***.116.132-**,143****,PROFESSOR DO MAGISTERIO SUPERIOR,C,00,null,003,-1,-1,Sem informação,-1,Sem informaç,null,26239000001515,NUCLEO DE DESENV AMAZONICO EM ENGENHARIA,26239,Universidade Federal do Pará,15000,Ministério da Educação,26239000000966,CENTRO DE PROCESSOS SELETIVOS,26239,Universidade Federal do Pará,15000,Ministério da Educação,2,Cargo,ATIVO PERMANENTE,null,null,REGIME JURIDICO UNICO,DEDICACAO EXCLUSIVA,01/03/2013,null,24/07/2006,2475,16/08/2006,null,PORTARIA,PORTARIA,-1
2868174,AARAO MEIR SERRUYA,***.693.832-**,331****,FISIOTERAPEUTA - 30H,S,00,null,201,-1,-1,Sem informação,-1,Sem informaç,null,26443035000000,COMPLEXO HOSPITALAR DA UFPA,26443,Empresa Brasileira de Serviços Hospitalares,15000,Ministério da Educação,-3,Inválido,26443,Empresa Brasileira de Serviços Hospitalares,15000,Ministério da Educação,2,Cargo,CELETISTA/EMPREGADO,null,null,CONSOLIDACAO DAS LEIS DO TRABALHO,30 HORAS SEMANAIS,07/12/2022,null,07/12/2022,null,null,null,CONTRATO,Inválido,-1
3137242,AARAO PEREIRA DE ARAUJO JUNIOR,***.031.184-**,027****,PROFESSOR ENS BASICO TECN TECNOLOGICO,null,-3,-3,-3,-1,-1,Sem informação,-1,Sem informaç,null,26417000000013,DIR. DESENVOLVIMENTO ENSINO-JP,26417,Instituto Federal da Paraíba,15000,Ministério da Educação,26417000000240,"UNID. ACAD. I DES, ESTR.M.AMB-JP",26417,Instituto Federal da Paraíba,15000,Ministério da Educação,2,Cargo,ATIVO PERMANENTE,null,null,REGIME JURIDICO UNICO,DEDICACAO EXCLUSIVA,01/03/2013,null,29/12/2008,000000124,03/05/1993,null,LEI,PORTARIA,PB


In [0]:
df_servidores_remuneracao = spark.read.option("header", True).option("sep", ";").option("encoding", "latin1").csv("dbfs:/Volumes/servidores/staging/dadosabertos/ativa/202510_Remuneracao.csv")
display(df_servidores_remuneracao.limit(5))


ANO,MES,Id_SERVIDOR_PORTAL,CPF,NOME,REMUNERAÇÃO BÁSICA BRUTA (R$),REMUNERAÇÃO BÁSICA BRUTA (U$),ABATE-TETO (R$),ABATE-TETO (U$),GRATIFICAÇÃO NATALINA (R$),GRATIFICAÇÃO NATALINA (U$),ABATE-TETO DA GRATIFICAÇÃO NATALINA (R$),ABATE-TETO DA GRATIFICAÇÃO NATALINA (U$),FÉRIAS (R$),FÉRIAS (U$),OUTRAS REMUNERAÇÕES EVENTUAIS (R$),OUTRAS REMUNERAÇÕES EVENTUAIS (U$),IRRF (R$),IRRF (U$),PSS/RPGS (R$),PSS/RPGS (U$),DEMAIS DEDUÇÕES (R$),DEMAIS DEDUÇÕES (U$),PENSÃO MILITAR (R$),PENSÃO MILITAR (U$),FUNDO DE SAÚDE (R$),FUNDO DE SAÚDE (U$),TAXA DE OCUPAÇÃO IMÓVEL FUNCIONAL (R$),TAXA DE OCUPAÇÃO IMÓVEL FUNCIONAL (U$),REMUNERAÇÃO APÓS DEDUÇÕES OBRIGATÓRIAS (R$),REMUNERAÇÃO APÓS DEDUÇÕES OBRIGATÓRIAS (U$),VERBAS INDENIZATÓRIAS REGISTRADAS EM SISTEMAS DE PESSOAL - CIVIL (R$)(*),VERBAS INDENIZATÓRIAS REGISTRADAS EM SISTEMAS DE PESSOAL - CIVIL (U$)(*),VERBAS INDENIZATÓRIAS REGISTRADAS EM SISTEMAS DE PESSOAL - MILITAR (R$)(*),VERBAS INDENIZATÓRIAS REGISTRADAS EM SISTEMAS DE PESSOAL - MILITAR (U$)(*),VERBAS INDENIZATÓRIAS PROGRAMA DESLIGAMENTO VOLUNTÁRIO  MP 792/2017 (R$),VERBAS INDENIZATÓRIAS PROGRAMA DESLIGAMENTO VOLUNTÁRIO  MP 792/2017 (U$),TOTAL DE VERBAS INDENIZATÓRIAS (R$)(*),TOTAL DE VERBAS INDENIZATÓRIAS (U$)(*)
2025,10,3174964,***.017.623-**,AARAO CARLOS LUZ MACAMBIRA,"12577,90","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","-2112,23","0,00","-1592,59","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","8873,08","0,00","1294,96","0,00","0,00","0,00","0,00","0,00","1294,96","0,00"
2025,10,2903139,***.116.132-**,AARAO FERREIRA LIMA NETO,"28342,89","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","803,52","0,00","-6267,89","0,00","-3049,58","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","19828,94","0,00","1000,00","0,00","0,00","0,00","0,00","0,00","1000,00","0,00"
2025,10,3137242,***.031.184-**,AARAO PEREIRA DE ARAUJO JUNIOR,"25379,42","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","3677,00","0,00","-5966,33","0,00","-3677,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","19413,09","0,00","1415,12","0,00","0,00","0,00","0,00","0,00","1415,12","0,00"
2025,10,3205874,***.859.807-**,AARAO SOARES,"7005,97","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","-769,05","0,00","-715,34","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","5521,58","0,00","9907,45","0,00","0,00","0,00","0,00","0,00","9907,45","0,00"
2025,10,3449428,***.086.942-**,AARAO TEIXEIRA DOS SANTOS,"6753,20","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","-740,76","0,00","-755,03","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","5257,41","0,00","1542,24","0,00","0,00","0,00","0,00","0,00","1542,24","0,00"


In [0]:
df_aposentados_cadastro = spark.read.option("header", True).option("sep", ";").option("encoding", "latin1").csv("dbfs:/Volumes/servidores/staging/dadosabertos/aposentados/202510_Cadastro.csv")
display(df_aposentados_cadastro.limit(5))

Id_SERVIDOR_PORTAL,NOME,CPF,MATRICULA,COD_TIPO_APOSENTADORIA,TIPO_APOSENTADORIA,DATA_APOSENTADORIA,DESCRICAO_CARGO,COD_UORG_LOTACAO,UORG_LOTACAO,COD_ORG_LOTACAO,ORG_LOTACAO,COD_ORGSUP_LOTACAO,ORGSUP_LOTACAO,COD_TIPO_VINCULO,TIPO_VINCULO,SITUACAO_VINCULO,REGIME_JURIDICO,JORNADA_DE_TRABALHO,DATA_INGRESSO_CARGOFUNCAO,DATA_NOMEACAO_CARGOFUNCAO,DATA_INGRESSO_ORGAO,DOCUMENTO_INGRESSO_SERVICOPUBLICO,DATA_DIPLOMA_INGRESSO_SERVICOPUBLICO,DIPLOMA_INGRESSO_CARGOFUNCAO,DIPLOMA_INGRESSO_ORGAO,DIPLOMA_INGRESSO_SERVICOPUBLICO
2025221,AARAO DE ANDRADE LIMA,***.559.144-**,033****,01,APOSENTADORIA VOLUNTARIA,12/11/2012,PROFESSOR DO MAGISTERIO SUPERIOR,-3,Inválido,26252,Universidade Federal de Campina Grande - PB,15000,Ministério da Educação,5,Aposentadoria,APOSENTADO,REGIME JURIDICO UNICO,DEDICACAO EXCLUSIVA,01/03/2013,null,10/04/2002,SN,10/03/1977,null,LEI,PORTARIA
3594060,AARAO MOREIRA DA SILVA,***.924.486-**,048****,01,APOSENTADORIA VOLUNTARIA,16/05/2011,AGENTE DE SAUDE PUBLICA,-3,Inválido,25000,Ministério da Saúde,-1,Sem informação,5,Aposentadoria,APOSENTADO,REGIME JURIDICO UNICO,40 HORAS SEMANAIS,01/03/2006,null,29/06/2010,SN,06/10/1975,null,PORTARIA,CONTRATO
87244,ABA ISRAEL COHEN PERSIANO,***.681.016-**,032****,01,APOSENTADORIA VOLUNTARIA,05/05/2014,PROFESSOR DO MAGISTERIO SUPERIOR,-3,Inválido,26238,Universidade Federal de Minas Gerais,15000,Ministério da Educação,5,Aposentadoria,APOSENTADO,REGIME JURIDICO UNICO,DEDICACAO EXCLUSIVA,29/01/1997,null,29/03/1978,0000000SN,18/04/1974,null,PORTARIA,PORTARIA
661149,ABADIA APARECIDA FAUSTINO,***.430.802-**,108****,01,APOSENTADORIA VOLUNTARIA,19/12/2024,AUXILIAR OPERACIONAL SERV DIVERSOS - NA,-3,Inválido,40806,DEP.DE CENTRAL.SERV.DE INATIVOS E PENS.,17500,MIN GESTAO E INOV EM SERV PUBLICOS,5,Aposentadoria,APOSENTADO,REGIME JURIDICO UNICO,40 HORAS SEMANAIS,01/08/1984,null,19/12/2024,NI,01/08/1984,null,PORTARIA,CONTRATO
3336973,ABADIA BELCHIOR GOMES,***.078.406-**,041****,01,APOSENTADORIA VOLUNTARIA,30/11/2012,SERVENTE DE LIMPEZA,-3,Inválido,26274,Fundação Universidade Federal Uberlândia,15000,Ministério da Educação,5,Aposentadoria,APOSENTADO,REGIME JURIDICO UNICO,40 HORAS SEMANAIS,01/03/2005,null,01/02/1982,000000S/N,01/02/1982,null,CONTRATO,CONTRATO


In [0]:

df_aposentados_remuneracao = spark.read.option("header", True).option("sep", ";").option("encoding", "latin1").csv("dbfs:/Volumes/servidores/staging/dadosabertos/aposentados/202510_Remuneracao.csv")
display(df_aposentados_remuneracao.limit(5))

ANO,MES,Id_SERVIDOR_PORTAL,CPF,NOME,REMUNERAÇÃO BÁSICA BRUTA (R$),REMUNERAÇÃO BÁSICA BRUTA (U$),ABATE-TETO (R$),ABATE-TETO (U$),GRATIFICAÇÃO NATALINA (R$),GRATIFICAÇÃO NATALINA (U$),ABATE-TETO DA GRATIFICAÇÃO NATALINA (R$),ABATE-TETO DA GRATIFICAÇÃO NATALINA (U$),FÉRIAS (R$),FÉRIAS (U$),OUTRAS REMUNERAÇÕES EVENTUAIS (R$),OUTRAS REMUNERAÇÕES EVENTUAIS (U$),IRRF (R$),IRRF (U$),PSS/RPGS (R$),PSS/RPGS (U$),DEMAIS DEDUÇÕES (R$),DEMAIS DEDUÇÕES (U$),PENSÃO MILITAR (R$),PENSÃO MILITAR (U$),FUNDO DE SAÚDE (R$),FUNDO DE SAÚDE (U$),TAXA DE OCUPAÇÃO IMÓVEL FUNCIONAL (R$),TAXA DE OCUPAÇÃO IMÓVEL FUNCIONAL (U$),REMUNERAÇÃO APÓS DEDUÇÕES OBRIGATÓRIAS (R$),REMUNERAÇÃO APÓS DEDUÇÕES OBRIGATÓRIAS (U$),VERBAS INDENIZATÓRIAS REGISTRADAS EM SISTEMAS DE PESSOAL - CIVIL (R$)(*),VERBAS INDENIZATÓRIAS REGISTRADAS EM SISTEMAS DE PESSOAL - CIVIL (U$)(*),VERBAS INDENIZATÓRIAS REGISTRADAS EM SISTEMAS DE PESSOAL - MILITAR (R$)(*),VERBAS INDENIZATÓRIAS REGISTRADAS EM SISTEMAS DE PESSOAL - MILITAR (U$)(*),VERBAS INDENIZATÓRIAS PROGRAMA DESLIGAMENTO VOLUNTÁRIO  MP 792/2017 (R$),VERBAS INDENIZATÓRIAS PROGRAMA DESLIGAMENTO VOLUNTÁRIO  MP 792/2017 (U$),TOTAL DE VERBAS INDENIZATÓRIAS (R$)(*),TOTAL DE VERBAS INDENIZATÓRIAS (U$)(*)
2025,10,2025221,***.559.144-**,AARAO DE ANDRADE LIMA,"25227,84","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","323,26","0,00","-4710,59","0,00","-2700,37","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","18140,14","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00"
2025,10,3594060,***.924.486-**,AARAO MOREIRA DA SILVA,"5782,26","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","-96,50","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","5685,76","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00"
2025,10,87244,***.681.016-**,ABA ISRAEL COHEN PERSIANO,"26113,50","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","-2846,51","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","23266,99","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00"
2025,10,661149,***.430.802-**,ABADIA APARECIDA FAUSTINO,"2519,19","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","2519,19","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00"
2025,10,3336973,***.078.406-**,ABADIA BELCHIOR GOMES,"4075,23","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","321,04","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","4396,27","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00"


Padronização dos nomes das colunas dos DataFrames para garantir compatibilidade com o Delta Lake

In [0]:
# Funções de normalização de nomes de colunas:
# - normalizar_nome_colunas: padroniza nomes removendo acentuação, espaços, caracteres especiais, substitui R$ por BRL e U$ por USD, limita a 64 caracteres e ajusta termos específicos.
# - colunas_a_normalizar: retorna pares (original, normalizado) para colunas que serão modificadas.

import unicodedata
import re

def normalizar_nome_colunas(col):
    col = col.replace('R$', 'BRL').replace('U$', 'USD')
    col = col.replace('$', '')
    col = unicodedata.normalize('NFKD', col).encode('ASCII', 'ignore').decode()
    col = re.sub(r'[ ,;{}()\n\t=*]', '_', col)
    col = re.sub(r'_+', '_', col)
    col = col.strip('_')
    col = col.replace('VERBAS_INDENIZATORIAS', 'INDENIZACAO')
    return col[:64]

def colunas_a_normalizar(df):
    return [
        (col, normalizar_nome_colunas(col))
        for col in df.columns
        if normalizar_nome_colunas(col) != col
    ]



Verificação dos DataFrames cujos nomes de colunas serão normalizados

In [0]:
colunas_a_normalizar(df_servidores_cadastro)

[]

In [0]:
colunas_a_normalizar(df_aposentados_cadastro)

[]

In [0]:
colunas_a_normalizar(df_servidores_remuneracao)

[('REMUNERAÇÃO BÁSICA BRUTA (R$)', 'REMUNERACAO_BASICA_BRUTA_BRL'),
 ('REMUNERAÇÃO BÁSICA BRUTA (U$)', 'REMUNERACAO_BASICA_BRUTA_USD'),
 ('ABATE-TETO (R$)', 'ABATE-TETO_BRL'),
 ('ABATE-TETO (U$)', 'ABATE-TETO_USD'),
 ('GRATIFICAÇÃO NATALINA (R$)', 'GRATIFICACAO_NATALINA_BRL'),
 ('GRATIFICAÇÃO NATALINA (U$)', 'GRATIFICACAO_NATALINA_USD'),
 ('ABATE-TETO DA GRATIFICAÇÃO NATALINA (R$)',
  'ABATE-TETO_DA_GRATIFICACAO_NATALINA_BRL'),
 ('ABATE-TETO DA GRATIFICAÇÃO NATALINA (U$)',
  'ABATE-TETO_DA_GRATIFICACAO_NATALINA_USD'),
 ('FÉRIAS (R$)', 'FERIAS_BRL'),
 ('FÉRIAS (U$)', 'FERIAS_USD'),
 ('OUTRAS REMUNERAÇÕES EVENTUAIS (R$)', 'OUTRAS_REMUNERACOES_EVENTUAIS_BRL'),
 ('OUTRAS REMUNERAÇÕES EVENTUAIS (U$)', 'OUTRAS_REMUNERACOES_EVENTUAIS_USD'),
 ('IRRF (R$)', 'IRRF_BRL'),
 ('IRRF (U$)', 'IRRF_USD'),
 ('PSS/RPGS (R$)', 'PSS/RPGS_BRL'),
 ('PSS/RPGS (U$)', 'PSS/RPGS_USD'),
 ('DEMAIS DEDUÇÕES (R$)', 'DEMAIS_DEDUCOES_BRL'),
 ('DEMAIS DEDUÇÕES (U$)', 'DEMAIS_DEDUCOES_USD'),
 ('PENSÃO MILITAR (R$)', 'PE

In [0]:
colunas_a_normalizar(df_aposentados_remuneracao)

[('REMUNERAÇÃO BÁSICA BRUTA (R$)', 'REMUNERACAO_BASICA_BRUTA_BRL'),
 ('REMUNERAÇÃO BÁSICA BRUTA (U$)', 'REMUNERACAO_BASICA_BRUTA_USD'),
 ('ABATE-TETO (R$)', 'ABATE-TETO_BRL'),
 ('ABATE-TETO (U$)', 'ABATE-TETO_USD'),
 ('GRATIFICAÇÃO NATALINA (R$)', 'GRATIFICACAO_NATALINA_BRL'),
 ('GRATIFICAÇÃO NATALINA (U$)', 'GRATIFICACAO_NATALINA_USD'),
 ('ABATE-TETO DA GRATIFICAÇÃO NATALINA (R$)',
  'ABATE-TETO_DA_GRATIFICACAO_NATALINA_BRL'),
 ('ABATE-TETO DA GRATIFICAÇÃO NATALINA (U$)',
  'ABATE-TETO_DA_GRATIFICACAO_NATALINA_USD'),
 ('FÉRIAS (R$)', 'FERIAS_BRL'),
 ('FÉRIAS (U$)', 'FERIAS_USD'),
 ('OUTRAS REMUNERAÇÕES EVENTUAIS (R$)', 'OUTRAS_REMUNERACOES_EVENTUAIS_BRL'),
 ('OUTRAS REMUNERAÇÕES EVENTUAIS (U$)', 'OUTRAS_REMUNERACOES_EVENTUAIS_USD'),
 ('IRRF (R$)', 'IRRF_BRL'),
 ('IRRF (U$)', 'IRRF_USD'),
 ('PSS/RPGS (R$)', 'PSS/RPGS_BRL'),
 ('PSS/RPGS (U$)', 'PSS/RPGS_USD'),
 ('DEMAIS DEDUÇÕES (R$)', 'DEMAIS_DEDUCOES_BRL'),
 ('DEMAIS DEDUÇÕES (U$)', 'DEMAIS_DEDUCOES_USD'),
 ('PENSÃO MILITAR (R$)', 'PE

### Cadastro dos Servidores Ativos

Os dados cadastrais dos servidores ativos foram armazenados em uma Delta Table sem necessidade de normalização dos nomes das colunas.

In [0]:
# Criação da Delta Table 
df_servidores_cadastro.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("servidores.bronze.servidores_cadastro")

In [0]:
%sql
select * from servidores.bronze.servidores_cadastro limit 5

Id_SERVIDOR_PORTAL,NOME,CPF,MATRICULA,DESCRICAO_CARGO,CLASSE_CARGO,REFERENCIA_CARGO,PADRAO_CARGO,NIVEL_CARGO,SIGLA_FUNCAO,NIVEL_FUNCAO,FUNCAO,CODIGO_ATIVIDADE,ATIVIDADE,OPCAO_PARCIAL,COD_UORG_LOTACAO,UORG_LOTACAO,COD_ORG_LOTACAO,ORG_LOTACAO,COD_ORGSUP_LOTACAO,ORGSUP_LOTACAO,COD_UORG_EXERCICIO,UORG_EXERCICIO,COD_ORG_EXERCICIO,ORG_EXERCICIO,COD_ORGSUP_EXERCICIO,ORGSUP_EXERCICIO,COD_TIPO_VINCULO,TIPO_VINCULO,SITUACAO_VINCULO,DATA_INICIO_AFASTAMENTO,DATA_TERMINO_AFASTAMENTO,REGIME_JURIDICO,JORNADA_DE_TRABALHO,DATA_INGRESSO_CARGOFUNCAO,DATA_NOMEACAO_CARGOFUNCAO,DATA_INGRESSO_ORGAO,DOCUMENTO_INGRESSO_SERVICOPUBLICO,DATA_DIPLOMA_INGRESSO_SERVICOPUBLICO,DIPLOMA_INGRESSO_CARGOFUNCAO,DIPLOMA_INGRESSO_ORGAO,DIPLOMA_INGRESSO_SERVICOPUBLICO,UF_EXERCICIO
1242227,FABRICIA MARIA DIAMANTINO CORREA,***.212.108-**,198****,PSICOLOGO-AREA,null,-3,-3,-3,-1,-1,Sem informação,-1,Sem informaç,null,26410000000094,DIRETORIA DE ASSUNTOS ESTUDANTIS - REI,26410,Instituto Federal do Norte de Minas Gerais,15000,Ministério da Educação,26410000000094,DIRETORIA DE ASSUNTOS ESTUDANTIS - REI,26410,Instituto Federal do Norte de Minas Gerais,15000,Ministério da Educação,2,Cargo,ATIVO PERMANENTE,null,null,REGIME JURIDICO UNICO,40 HORAS SEMANAIS,21/11/2012,null,11/10/2012,478,08/11/2012,null,PORTARIA,PORTARIA,MG
1582275,FABRICIA MARTINS SALES,***.724.437-**,188****,PROFESSOR ENS BASICO TECN TECNOLOGICO,C,00,null,004,-1,-1,Sem informação,-1,Sem informaç,null,26434000000021,DIRETORIA GERAL DO CAMPUS CAMPOS-GUARUS,26434,Instituto Federal Fluminense,15000,Ministério da Educação,26434000000556,COORDENACAO DO CURSO DE ENFERMAGEM,26434,Instituto Federal Fluminense,15000,Ministério da Educação,2,Cargo,ATIVO PERMANENTE,null,null,REGIME JURIDICO UNICO,DEDICACAO EXCLUSIVA,01/03/2013,null,28/07/2011,464,03/08/2011,null,PORTARIA,PORTARIA,RJ
45156,FABRICIA MATTE CAYE,***.870.472-**,210****,ECONOMISTA,E,00,014,000,-1,-1,Sem informação,-1,Sem informaç,null,26437000000314,NUCLEO DE RELACOES MUNDO DO TRABALHO,26437,Instituto Federal de Roraima,15000,Ministério da Educação,26437000000314,NUCLEO DE RELACOES MUNDO DO TRABALHO,26437,Instituto Federal de Roraima,15000,Ministério da Educação,2,Cargo,ATIVO PERMANENTE,null,null,REGIME JURIDICO UNICO,40 HORAS SEMANAIS,03/04/2014,null,07/03/2014,272,03/04/2014,null,PORTARIA,PORTARIA,RR
1899747,FABRICIA MAYARA GALVAO RAFAEL MEDEIROS,***.904.634-**,331****,TECNICO EM ENFERMAGEM - 36H,T,00,null,101,-1,-1,Sem informação,-1,Sem informaç,null,26443012000000,HOSPITAL UNIVERSITARIO ANA BEZERRA,26443,Empresa Brasileira de Serviços Hospitalares,15000,Ministério da Educação,-3,Inválido,26443,Empresa Brasileira de Serviços Hospitalares,15000,Ministério da Educação,2,Cargo,CELETISTA/EMPREGADO,null,null,CONSOLIDACAO DAS LEIS DO TRABALHO,36 HORAS SEMANAIS,06/12/2022,null,06/12/2022,null,null,null,CONTRATO,Inválido,-1
1194443,FABRICIA MICHELE DA SILVA SOLER FERNANDES,***.494.698-**,203****,MEDICO PROGRAMA MAIS MEDICO,1,00,I,000,-1,-1,Sem informação,-1,Sem informaç,null,25000000007895,DEPTO DE APOIO A GESTAO ATENCAO PRIMARIA,25000,Ministério da Saúde,-1,Sem informação,25000000008462,COORD-GERAL DE PROVIMENTO PROFISSIONAL,25000,Ministério da Saúde,-1,Sem informação,2,Cargo,RESIDENCIA E PMM,null,null,MEDICO - PROGRAMA MAIS MEDICO,40 HORAS SEMANAIS,07/07/2025,null,07/07/2025,SN,07/07/2025,null,CONTRATO,CONTRATO,DF


### Cadastro dos Aposentados

Os dados cadastrais dos aposentados foram armazenados em uma Delta Table sem necessidade de normalização dos nomes das colunas.

In [0]:
# Criação da Delta Table 
df_aposentados_cadastro.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("servidores.bronze.aposentados_cadastro")

In [0]:
%sql
select * from servidores.bronze.aposentados_cadastro limit 5

Id_SERVIDOR_PORTAL,NOME,CPF,MATRICULA,COD_TIPO_APOSENTADORIA,TIPO_APOSENTADORIA,DATA_APOSENTADORIA,DESCRICAO_CARGO,COD_UORG_LOTACAO,UORG_LOTACAO,COD_ORG_LOTACAO,ORG_LOTACAO,COD_ORGSUP_LOTACAO,ORGSUP_LOTACAO,COD_TIPO_VINCULO,TIPO_VINCULO,SITUACAO_VINCULO,REGIME_JURIDICO,JORNADA_DE_TRABALHO,DATA_INGRESSO_CARGOFUNCAO,DATA_NOMEACAO_CARGOFUNCAO,DATA_INGRESSO_ORGAO,DOCUMENTO_INGRESSO_SERVICOPUBLICO,DATA_DIPLOMA_INGRESSO_SERVICOPUBLICO,DIPLOMA_INGRESSO_CARGOFUNCAO,DIPLOMA_INGRESSO_ORGAO,DIPLOMA_INGRESSO_SERVICOPUBLICO
2320436,JOSE ANTONIO OLIVEIRA PERBELINI LEMENHE,***.644.418-**,028****,01,APOSENTADORIA VOLUNTARIA,19/10/2007,PROFESSOR DO MAGISTERIO SUPERIOR,-3,Inválido,26233,Universidade Federal do Ceará,15000,Ministério da Educação,5,Aposentadoria,APOSENTADO,REGIME JURIDICO UNICO,DEDICACAO EXCLUSIVA,01/03/2013,null,21/09/1970,000000001,21/09/1970,null,PORTARIA,PORTARIA
603649,JOSE ANTONIO OLIVEIRA RIBEIRO,***.789.003-**,109****,01,APOSENTADORIA VOLUNTARIA,29/03/2018,VIGILANTE,-3,Inválido,26272,Fundação Universidade Federal do Maranhão,15000,Ministério da Educação,5,Aposentadoria,APOSENTADO,REGIME JURIDICO UNICO,40 HORAS SEMANAIS,01/08/2004,null,18/11/1994,775/94-GR,06/12/1994,null,PORTARIA,PORTARIA
752316,JOSE ANTONIO ORTEGA,***.177.167-**,037****,01,APOSENTADORIA VOLUNTARIA,08/09/1997,PROFESSOR DO MAGISTERIO SUPERIOR,-3,Inválido,26245,Universidade Federal do Rio de Janeiro,15000,Ministério da Educação,5,Aposentadoria,APOSENTADO,REGIME JURIDICO UNICO,DEDICACAO EXCLUSIVA,01/03/2013,null,30/04/1975,7596,01/01/1971,null,LEI,LEI
952341,JOSE ANTONIO OUTEIRO LOCHE,***.385.808-**,019****,01,APOSENTADORIA VOLUNTARIA,19/12/2019,CONTROLADOR DE TRAFEGO AEREO,-3,Inválido,21000,Comando da Aeronáutica,40115,MINISTERIO DA DEFESA,5,Aposentadoria,APOSENTADO,REGIME JURIDICO UNICO,40 HORAS SEMANAIS,18/04/1977,null,18/04/1970,BOL 070,18/04/1977,null,BOLETIM INTERNO,BOLETIM INTERNO
300112,JOSE ANTONIO PACHECCO,***.528.648-**,122****,01,APOSENTADORIA VOLUNTARIA,11/03/2013,AUDITOR-FISCAL DA RECEITA FEDERAL BRASIL,-3,Inválido,40806,DEP.DE CENTRAL.SERV.DE INATIVOS E PENS.,17500,MIN GESTAO E INOV EM SERV PUBLICOS,5,Aposentadoria,APOSENTADO,REGIME JURIDICO UNICO,40 HORAS SEMANAIS,30/12/2016,null,10/09/2018,344,18/08/1997,null,DECRETO,PORTARIA


### Remuneração dos Servidores Ativos

Os dados de remuneração dos servidores ativos foram normalizados e armazenados em uma Delta Table


In [0]:
# Padronização dos nomes das colunas 
new_columns_servidores_remuneracao = [normalizar_nome_colunas(col) for col in df_servidores_remuneracao.columns]
df_servidores_remuneracao = df_servidores_remuneracao.toDF(*new_columns_servidores_remuneracao)

# Criação da Delta Table 
df_servidores_remuneracao.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("servidores.bronze.servidores_remuneracao")

In [0]:
%sql
select * from servidores.bronze.servidores_remuneracao limit 5

ANO,MES,Id_SERVIDOR_PORTAL,CPF,NOME,REMUNERACAO_BASICA_BRUTA_BRL,REMUNERACAO_BASICA_BRUTA_USD,ABATE-TETO_BRL,ABATE-TETO_USD,GRATIFICACAO_NATALINA_BRL,GRATIFICACAO_NATALINA_USD,ABATE-TETO_DA_GRATIFICACAO_NATALINA_BRL,ABATE-TETO_DA_GRATIFICACAO_NATALINA_USD,FERIAS_BRL,FERIAS_USD,OUTRAS_REMUNERACOES_EVENTUAIS_BRL,OUTRAS_REMUNERACOES_EVENTUAIS_USD,IRRF_BRL,IRRF_USD,PSS/RPGS_BRL,PSS/RPGS_USD,DEMAIS_DEDUCOES_BRL,DEMAIS_DEDUCOES_USD,PENSAO_MILITAR_BRL,PENSAO_MILITAR_USD,FUNDO_DE_SAUDE_BRL,FUNDO_DE_SAUDE_USD,TAXA_DE_OCUPACAO_IMOVEL_FUNCIONAL_BRL,TAXA_DE_OCUPACAO_IMOVEL_FUNCIONAL_USD,REMUNERACAO_APOS_DEDUCOES_OBRIGATORIAS_BRL,REMUNERACAO_APOS_DEDUCOES_OBRIGATORIAS_USD,INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_CIVIL_BRL,INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_CIVIL_USD,INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_MILITAR_BRL,INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_MILITAR_USD,INDENIZACAO_PROGRAMA_DESLIGAMENTO_VOLUNTARIO_MP_792/2017_BRL,INDENIZACAO_PROGRAMA_DESLIGAMENTO_VOLUNTARIO_MP_792/2017_USD,TOTAL_DE_INDENIZACAO_BRL,TOTAL_DE_INDENIZACAO_USD
2025,10,1970910,***.286.172-**,RAIMUNDO DELGADO MARTINS,"3528,43","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","1000,00","0,00","-44,02","0,00","-316,81","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","4167,60","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00"
2025,10,1146782,***.235.172-**,RAIMUNDO DIAS DA SILVA,"7152,89","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","7400,07","0,00","-2830,22","0,00","-767,16","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","10955,58","0,00","1000,00","0,00","0,00","0,00","0,00","0,00","1000,00","0,00"
2025,10,3248864,***.054.702-**,RAIMUNDO DIAS DE CARVALHO,"19049,66","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","2141,94","0,00","-4277,77","0,00","-2141,94","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","14771,89","0,00","1553,25","0,00","0,00","0,00","0,00","0,00","1553,25","0,00"
2025,10,2367589,***.207.071-**,RAIMUNDO DIAS DOS SANTOS,"8882,16","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","-1336,75","0,00","-716,76","0,00","-22,68","0,00","0,00","0,00","0,00","0,00","0,00","0,00","6805,97","0,00","1249,51","0,00","0,00","0,00","0,00","0,00","1249,51","0,00"
2025,10,2095806,***.202.963-**,RAIMUNDO DIAS GOMES,"12580,25","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","1445,38","0,00","-1389,73","0,00","-1445,38","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","11190,52","0,00","1397,86","0,00","0,00","0,00","0,00","0,00","1397,86","0,00"


### Remuneração dos Servidores Aposentados

Os dados de remuneração dos servidores aposentados foram normalizados e armazenados em uma Delta Table

In [0]:

# Padronização dos nomes das colunas 
new_columns_aposentados_remumeracao = [normalizar_nome_colunas(col) for col in df_aposentados_remuneracao.columns]

df_aposentados_remuneracao = df_aposentados_remuneracao.toDF(*new_columns_aposentados_remumeracao)

# Criação da Delta Table 
df_aposentados_remuneracao.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("servidores.bronze.aposentados_remuneracao")


In [0]:
%sql
select * from servidores.bronze.aposentados_remuneracao limit 5

ANO,MES,Id_SERVIDOR_PORTAL,CPF,NOME,REMUNERACAO_BASICA_BRUTA_BRL,REMUNERACAO_BASICA_BRUTA_USD,ABATE-TETO_BRL,ABATE-TETO_USD,GRATIFICACAO_NATALINA_BRL,GRATIFICACAO_NATALINA_USD,ABATE-TETO_DA_GRATIFICACAO_NATALINA_BRL,ABATE-TETO_DA_GRATIFICACAO_NATALINA_USD,FERIAS_BRL,FERIAS_USD,OUTRAS_REMUNERACOES_EVENTUAIS_BRL,OUTRAS_REMUNERACOES_EVENTUAIS_USD,IRRF_BRL,IRRF_USD,PSS/RPGS_BRL,PSS/RPGS_USD,DEMAIS_DEDUCOES_BRL,DEMAIS_DEDUCOES_USD,PENSAO_MILITAR_BRL,PENSAO_MILITAR_USD,FUNDO_DE_SAUDE_BRL,FUNDO_DE_SAUDE_USD,TAXA_DE_OCUPACAO_IMOVEL_FUNCIONAL_BRL,TAXA_DE_OCUPACAO_IMOVEL_FUNCIONAL_USD,REMUNERACAO_APOS_DEDUCOES_OBRIGATORIAS_BRL,REMUNERACAO_APOS_DEDUCOES_OBRIGATORIAS_USD,INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_CIVIL_BRL,INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_CIVIL_USD,INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_MILITAR_BRL,INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_MILITAR_USD,INDENIZACAO_PROGRAMA_DESLIGAMENTO_VOLUNTARIO_MP_792/2017_BRL,INDENIZACAO_PROGRAMA_DESLIGAMENTO_VOLUNTARIO_MP_792/2017_USD,TOTAL_DE_INDENIZACAO_BRL,TOTAL_DE_INDENIZACAO_USD
2025,10,711607,***.876.306-**,LUCIO BATITUCCI COSTA,"29760,95","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","161,63","0,00","0,00","0,00","-3493,89","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","26428,69","0,00","2834,99","0,00","0,00","0,00","0,00","0,00","2834,99","0,00"
2025,10,1444858,***.424.147-**,LUCIO BERNARDO CARNEIRO,"17304,13","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","-2943,24","0,00","-1392,96","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","12967,93","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00"
2025,10,2643558,***.340.527-**,LUCIO BERNARDO DA SILVA,"11163,23","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","422,72","0,00","-1470,58","0,00","-435,84","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","9679,53","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00"
2025,10,2988735,***.690.677-**,LUCIO CAMPOS DA SILVA,"22910,69","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","-2318,04","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","20592,65","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00"
2025,10,965884,***.510.332-**,LUCIO CAMPOS DOS SANTOS,"18771,76","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","3402,03","0,00","-2700,14","0,00","-1971,03","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","17502,62","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00"


# 5.2) Camada Silver

Na camada Silver, os dados brutos da Bronze passam por processos de limpeza, filtragem e padronização para garantir maior qualidade e consistência.  
São selecionadas apenas as colunas relevantes para análise, removendo informações sensíveis ou desnecessárias.  
Valores nulos e inconsistências são tratados, tipos de dados são ajustados e regras de negócio são aplicadas para preparar os dados para agregações, cruzamentos e análises exploratórias nas etapas seguintes.

In [0]:
%sql
USE CATALOG servidores;
USE SCHEMA silver;

## 1 - Cópia das tabelas da camada bronze para silver

In [0]:
%sql
-- Copia servidores_cadastro da bronze para silver
CREATE OR REPLACE TABLE servidores.silver.servidores_cadastro AS
SELECT * FROM servidores.bronze.servidores_cadastro;

-- Copia aposentados_cadastro da bronze para silver
CREATE OR REPLACE TABLE servidores.silver.aposentados_cadastro AS
SELECT * FROM servidores.bronze.aposentados_cadastro;

-- Copia servidores_remuneracao da bronze para silver
CREATE OR REPLACE TABLE servidores.silver.servidores_remuneracao AS
SELECT * FROM servidores.bronze.servidores_remuneracao;

-- Copia aposentados_remuneracao da bronze para silver
CREATE OR REPLACE TABLE servidores.silver.aposentados_remuneracao AS
SELECT * FROM servidores.bronze.aposentados_remuneracao;

num_affected_rows,num_inserted_rows


## 2 - Processos de análise, filtragem e tratamento dos dados nas tabelas de cadastro de servidores ativos e aposentados



### `servidores_cadastro`

- **Seleção de colunas para análise e modelagem:**
    - `Id_SERVIDOR_PORTAL`: Identificador único do servidor, essencial para garantir unicidade e rastreabilidade dos registros.
    - `NOME`: Nome completo do servidor, necessário para identificação individual e validação de dados.
    - `DESCRICAO_CARGO`: Cargo ocupado, fundamental para análises de distribuição e remuneração por função.
    - `CODIGO_ATIVIDADE`, `ATIVIDADE`: Código e descrição da atividade, permitem segmentar servidores por área de atuação.
    - `COD_ORG_LOTACAO`, `ORG_LOTACAO`: Código e órgão de lotação, importantes para análises por unidade administrativa.
    - `COD_ORGSUP_LOTACAO`, `ORGSUP_LOTACAO`: Código e órgão superior, viabilizam agrupamentos por estrutura hierárquica.
    - `SITUACAO_VINCULO`: Situação do vínculo, permite identificar servidores ativos, afastados, licenciados, etc.
    - `DATA_INICIO_AFASTAMENTO`, `DATA_TERMINO_AFASTAMENTO`: Datas de afastamento, necessárias para análises de licenças e ausências.
    - `UF_EXERCICIO`: UF de exercício, relevante para segmentação geográfica dos servidores.
    - `cod_descricao`: Coluna criada para armazenar o código referente à descrição do cargo, garantindo integridade referencial e facilitando análises por função.
    - `cod_vinculo`: Coluna criada para armazenar o código referente ao vínculo do servidor, permitindo segmentação e análise dos diferentes tipos de vínculos.

- **Padronização e renomeação dos campos:**  
    - `Id_SERVIDOR_PORTAL` → `id`  
    - `NOME` → `nome`  
    - `DESCRICAO_CARGO` → `desc_cargo`  
    - `CODIGO_ATIVIDADE` → `cod_atividade`  
    - `ATIVIDADE` → `desc_atividade`  
    - `COD_ORG_LOTACAO` → `cod_org_lotacao`  
    - `ORG_LOTACAO` → `org_lotacao`  
    - `COD_ORGSUP_LOTACAO` → `cod_orgsup_lotacao`  
    - `ORGSUP_LOTACAO` → `orgsup_lotacao`  
    - `SITUACAO_VINCULO` → `situacao_vinculo`  
    - `DATA_INICIO_AFASTAMENTO` → `dt_inicio_afastamento`  
    - `DATA_TERMINO_AFASTAMENTO` → `dt_termino_afastamento`  
    - `UF_EXERCICIO` → `uf_exercicio`  
- Inclusão das colunas:
    - `cod_descricao`: código exclusivo para cada descrição de cargo, garantindo integridade referencial e facilitando análises agregadas por função.
    - `cod_vinculo`: código para cada tipo de vínculo, permitindo segmentação precisa dos servidores conforme vínculo funcional e aprimorando a rastreabilidade dos dados.

- **Definição dos campos obrigatórios e opcionais:**  
  - Os campos `id` e `nome` são obrigatórios e não podem conter valores nulos.
  - Os demais campos são opcionais e podem apresentar valores nulos, preservando a integridade e a representatividade dos dados originais.

- **Conversão e padronização de tipos:**  
    - Códigos convertidos para inteiro.  
    - Datas para formato `yyyy-MM-dd`.  
    - Demais campos mantidos como string.

- **Tratamento de valores nulos e ausentes:**  
    - Campos de datas de afastamento permanecem nulos, indicando ausência de registros no período.  
    - Outros campos podem ter nulos em casos de cadastro incompleto ou vínculos atípicos.
    - Campos `cod_descricao` e `cod_vinculo` mantêm nulos quando não há registro correspondente.

- **Validação dos dados:**  
    - Garantia de unicidade em `id`.  
    - Checagem de consistência entre códigos e descrições.  
    - Validação de integridade referencial entre códigos e suas respectivas descrições.
    - Verificação de ausência de duplicidades e registros inconsistentes.
    - Conferência de domínios esperados para atributos categóricos.

- **Sobreescrita da tabela tratada:**  
    - A tabela `servidores_cadastro` tratada é sobreescrita após o processo de limpeza, validação e padronização, garantindo que apenas registros válidos e consistentes sejam mantidos para análise.

- **Documentação e comentários no catálogo de dados:**  
    - Comentários foram adicionados para as colunas e para a tabela no catálogo de dados, detalhando o significado de cada atributo, incluindo as etapas de inclusão das colunas `cod_descricao` e `cod_vinculo`.

### • Seleção de colunas para análise e modelagem
Processo de escolha das colunas relevantes para garantir unicidade, rastreabilidade e representatividade dos dados dos servidores.

In [0]:
%sql
select * from servidores.silver.servidores_cadastro limit 5

Id_SERVIDOR_PORTAL,NOME,CPF,MATRICULA,DESCRICAO_CARGO,CLASSE_CARGO,REFERENCIA_CARGO,PADRAO_CARGO,NIVEL_CARGO,SIGLA_FUNCAO,NIVEL_FUNCAO,FUNCAO,CODIGO_ATIVIDADE,ATIVIDADE,OPCAO_PARCIAL,COD_UORG_LOTACAO,UORG_LOTACAO,COD_ORG_LOTACAO,ORG_LOTACAO,COD_ORGSUP_LOTACAO,ORGSUP_LOTACAO,COD_UORG_EXERCICIO,UORG_EXERCICIO,COD_ORG_EXERCICIO,ORG_EXERCICIO,COD_ORGSUP_EXERCICIO,ORGSUP_EXERCICIO,COD_TIPO_VINCULO,TIPO_VINCULO,SITUACAO_VINCULO,DATA_INICIO_AFASTAMENTO,DATA_TERMINO_AFASTAMENTO,REGIME_JURIDICO,JORNADA_DE_TRABALHO,DATA_INGRESSO_CARGOFUNCAO,DATA_NOMEACAO_CARGOFUNCAO,DATA_INGRESSO_ORGAO,DOCUMENTO_INGRESSO_SERVICOPUBLICO,DATA_DIPLOMA_INGRESSO_SERVICOPUBLICO,DIPLOMA_INGRESSO_CARGOFUNCAO,DIPLOMA_INGRESSO_ORGAO,DIPLOMA_INGRESSO_SERVICOPUBLICO,UF_EXERCICIO
3174964,AARAO CARLOS LUZ MACAMBIRA,***.017.623-**,167****,BIBLIOTECARIO-DOCUMENTALISTA,E,00,018,000,-1,-1,Sem informação,-1,Sem informaç,null,26405000000040,DIRETORIA GERAL/CAMPUS SOBRAL,26405,Instituto Federal do Ceará,15000,Ministério da Educação,26405000000717,DIRETORIA DE ENSINO-SOB,26405,Instituto Federal do Ceará,15000,Ministério da Educação,2,Cargo,ATIVO PERMANENTE,null,null,REGIME JURIDICO UNICO,40 HORAS SEMANAIS,11/02/2009,null,29/12/2008,699,11/02/2009,null,LEI,PORTARIA,CE
2903139,AARAO FERREIRA LIMA NETO,***.116.132-**,143****,Sem informaç,null,-1,-1,-1,CD,0003,CARGO DE DIRECAO - CD - IFES,0099,DIRETOR(A),S,26239000001515,NUCLEO DE DESENV AMAZONICO EM ENGENHARIA,26239,Universidade Federal do Pará,15000,Ministério da Educação,26239000000966,CENTRO DE PROCESSOS SELETIVOS,26239,Universidade Federal do Pará,15000,Ministério da Educação,1,Função,ATIVO PERMANENTE,null,null,REGIME JURIDICO UNICO,DEDICACAO EXCLUSIVA,23/11/2024,null,24/07/2006,2475,16/08/2006,null,PORTARIA,PORTARIA,PA
2903139,AARAO FERREIRA LIMA NETO,***.116.132-**,143****,PROFESSOR DO MAGISTERIO SUPERIOR,C,00,null,003,-1,-1,Sem informação,-1,Sem informaç,null,26239000001515,NUCLEO DE DESENV AMAZONICO EM ENGENHARIA,26239,Universidade Federal do Pará,15000,Ministério da Educação,26239000000966,CENTRO DE PROCESSOS SELETIVOS,26239,Universidade Federal do Pará,15000,Ministério da Educação,2,Cargo,ATIVO PERMANENTE,null,null,REGIME JURIDICO UNICO,DEDICACAO EXCLUSIVA,01/03/2013,null,24/07/2006,2475,16/08/2006,null,PORTARIA,PORTARIA,-1
2868174,AARAO MEIR SERRUYA,***.693.832-**,331****,FISIOTERAPEUTA - 30H,S,00,null,201,-1,-1,Sem informação,-1,Sem informaç,null,26443035000000,COMPLEXO HOSPITALAR DA UFPA,26443,Empresa Brasileira de Serviços Hospitalares,15000,Ministério da Educação,-3,Inválido,26443,Empresa Brasileira de Serviços Hospitalares,15000,Ministério da Educação,2,Cargo,CELETISTA/EMPREGADO,null,null,CONSOLIDACAO DAS LEIS DO TRABALHO,30 HORAS SEMANAIS,07/12/2022,null,07/12/2022,null,null,null,CONTRATO,Inválido,-1
3137242,AARAO PEREIRA DE ARAUJO JUNIOR,***.031.184-**,027****,PROFESSOR ENS BASICO TECN TECNOLOGICO,null,-3,-3,-3,-1,-1,Sem informação,-1,Sem informaç,null,26417000000013,DIR. DESENVOLVIMENTO ENSINO-JP,26417,Instituto Federal da Paraíba,15000,Ministério da Educação,26417000000240,"UNID. ACAD. I DES, ESTR.M.AMB-JP",26417,Instituto Federal da Paraíba,15000,Ministério da Educação,2,Cargo,ATIVO PERMANENTE,null,null,REGIME JURIDICO UNICO,DEDICACAO EXCLUSIVA,01/03/2013,null,29/12/2008,000000124,03/05/1993,null,LEI,PORTARIA,PB


In [0]:

df_servidores_cadastro = spark.table("servidores.silver.servidores_cadastro")

display(df_servidores_cadastro.columns)

_1
Id_SERVIDOR_PORTAL
NOME
CPF
MATRICULA
DESCRICAO_CARGO
CLASSE_CARGO
REFERENCIA_CARGO
PADRAO_CARGO
NIVEL_CARGO
SIGLA_FUNCAO


In [0]:
from pyspark.sql.functions import to_date

# Remoção de colunas que não agregam valor para análise:
# - Dados sensíveis (CPF, MATRÍCULA)
# - Detalhes excessivos de cargo, função, lotação e exercício que não são relevantes para agregações ou perguntas analíticas
# - Informações administrativas e documentos que não contribuem para as análises propostas
lista_colunas_remover = [
    'CPF', 'MATRICULA', 'CLASSE_CARGO', 'REFERENCIA_CARGO', 'PADRAO_CARGO', 'NIVEL_CARGO',
    'SIGLA_FUNCAO', 'NIVEL_FUNCAO', 'FUNCAO', "OPCAO_PARCIAL", 'COD_UORG_LOTACAO', 'UORG_LOTACAO',
    'COD_UORG_EXERCICIO', 'UORG_EXERCICIO', 'COD_ORG_EXERCICIO', 'ORG_EXERCICIO',
    'COD_ORGSUP_EXERCICIO', 'ORGSUP_EXERCICIO', 'REGIME_JURIDICO', 'JORNADA_DE_TRABALHO',
    'DATA_INGRESSO_CARGOFUNCAO', 'DATA_NOMEACAO_CARGOFUNCAO', 'DATA_INGRESSO_ORGAO',
    'DOCUMENTO_INGRESSO_SERVICOPUBLICO', 'DATA_DIPLOMA_INGRESSO_SERVICOPUBLICO',
    'DIPLOMA_INGRESSO_CARGOFUNCAO', 'DIPLOMA_INGRESSO_ORGAO', 'DIPLOMA_INGRESSO_SERVICOPUBLICO', 'COD_TIPO_VINCULO', 'TIPO_VINCULO'
]

df_filtrado_servidores_cadastro = df_servidores_cadastro.drop(*lista_colunas_remover)

# Mapeamento dos nomes nas colunas selecionadas para sintéticos
mapeamento_colunas = {
    "Id_SERVIDOR_PORTAL": "id",
    "NOME": "nome",
    "DESCRICAO_CARGO": "desc_cargo",
    "CODIGO_ATIVIDADE": "cod_atividade",
    "ATIVIDADE": "desc_atividade",
    "COD_ORG_LOTACAO": "cod_org_lotacao",
    "ORG_LOTACAO": "org_lotacao",
    "COD_ORGSUP_LOTACAO": "cod_orgsup_lotacao",
    "ORGSUP_LOTACAO": "orgsup_lotacao",
    "SITUACAO_VINCULO": "situacao_vinculo",
    "DATA_INICIO_AFASTAMENTO": "dt_inicio_afastamento",
    "DATA_TERMINO_AFASTAMENTO": "dt_fim_afastamento",
    "UF_EXERCICIO": "uf_exercicio"
}

# Renomeia as colunas conforme o mapeamento
for original, novo in mapeamento_colunas.items():
    if original in df_filtrado_servidores_cadastro.columns:
        df_filtrado_servidores_cadastro = df_filtrado_servidores_cadastro.withColumnRenamed(original, novo)

display(df_filtrado_servidores_cadastro.limit(5))

id,nome,desc_cargo,cod_atividade,desc_atividade,cod_org_lotacao,org_lotacao,cod_orgsup_lotacao,orgsup_lotacao,situacao_vinculo,dt_inicio_afastamento,dt_fim_afastamento,uf_exercicio
3174964,AARAO CARLOS LUZ MACAMBIRA,BIBLIOTECARIO-DOCUMENTALISTA,-1,Sem informaç,26405,Instituto Federal do Ceará,15000,Ministério da Educação,ATIVO PERMANENTE,null,null,CE
2903139,AARAO FERREIRA LIMA NETO,Sem informaç,0099,DIRETOR(A),26239,Universidade Federal do Pará,15000,Ministério da Educação,ATIVO PERMANENTE,null,null,PA
2903139,AARAO FERREIRA LIMA NETO,PROFESSOR DO MAGISTERIO SUPERIOR,-1,Sem informaç,26239,Universidade Federal do Pará,15000,Ministério da Educação,ATIVO PERMANENTE,null,null,-1
2868174,AARAO MEIR SERRUYA,FISIOTERAPEUTA - 30H,-1,Sem informaç,26443,Empresa Brasileira de Serviços Hospitalares,15000,Ministério da Educação,CELETISTA/EMPREGADO,null,null,-1
3137242,AARAO PEREIRA DE ARAUJO JUNIOR,PROFESSOR ENS BASICO TECN TECNOLOGICO,-1,Sem informaç,26417,Instituto Federal da Paraíba,15000,Ministério da Educação,ATIVO PERMANENTE,null,null,PB


### • Tratamento de valores nulos e ausentes
Processo de substituição de valores inválidos ou ausentes por None, conforme regras de negócio e padronização dos dados.

In [0]:
# Substituição de valores inválidos ou faltantes por none 
df_filtrado_servidores_cadastro = df_filtrado_servidores_cadastro.replace("-1", None)
df_filtrado_servidores_cadastro = df_filtrado_servidores_cadastro.replace(
    ["Sem informaç", "Inválido", "Sem informação", ""],
    None,
    subset=["desc_cargo", "orgsup_lotacao", "desc_atividade"]
)

display(df_filtrado_servidores_cadastro.limit(5))


id,nome,desc_cargo,cod_atividade,desc_atividade,cod_org_lotacao,org_lotacao,cod_orgsup_lotacao,orgsup_lotacao,situacao_vinculo,dt_inicio_afastamento,dt_fim_afastamento,uf_exercicio
3174964,AARAO CARLOS LUZ MACAMBIRA,BIBLIOTECARIO-DOCUMENTALISTA,null,null,26405,Instituto Federal do Ceará,15000,Ministério da Educação,ATIVO PERMANENTE,null,null,CE
2903139,AARAO FERREIRA LIMA NETO,null,0099,DIRETOR(A),26239,Universidade Federal do Pará,15000,Ministério da Educação,ATIVO PERMANENTE,null,null,PA
2903139,AARAO FERREIRA LIMA NETO,PROFESSOR DO MAGISTERIO SUPERIOR,null,null,26239,Universidade Federal do Pará,15000,Ministério da Educação,ATIVO PERMANENTE,null,null,null
2868174,AARAO MEIR SERRUYA,FISIOTERAPEUTA - 30H,null,null,26443,Empresa Brasileira de Serviços Hospitalares,15000,Ministério da Educação,CELETISTA/EMPREGADO,null,null,null
3137242,AARAO PEREIRA DE ARAUJO JUNIOR,PROFESSOR ENS BASICO TECN TECNOLOGICO,null,null,26417,Instituto Federal da Paraíba,15000,Ministério da Educação,ATIVO PERMANENTE,null,null,PB


### • Inclusão da coluna de código para o cargo do servidor
Processo de criação do identificador único para cada cargo, garantindo integridade referencial e facilitando análises por função.

In [0]:
from pyspark.sql.functions import col, dense_rank
from pyspark.sql.window import Window

# Cria janela ordenada por desc_cargo
window_spec = Window.orderBy(col("desc_cargo").asc())

# Gera código sequencial cod_cargo para cada valor distinto e não nulo de desc_cargo
df_contagem_desc_cargo = (
    df_filtrado_servidores_cadastro
    .filter(col("desc_cargo").isNotNull())
    .select("desc_cargo")
    .distinct()
    .withColumn(
        "cod_cargo",
        dense_rank().over(window_spec)
    )
    .orderBy(col("desc_cargo").asc())
)

display(df_contagem_desc_cargo.limit(5))

# Realiza o join para inserir cod_cargo e reordena as colunas para que cod_cargo fique antes de desc_cargo
colunas_originais = df_filtrado_servidores_cadastro.columns
colunas_sem_cod = [col for col in colunas_originais if col != "cod_cargo"]
pos_desc = colunas_sem_cod.index("desc_cargo")
colunas_reordenadas = (
    colunas_sem_cod[:pos_desc] +
    ["cod_cargo"] +
    colunas_sem_cod[pos_desc:]
)
df_filtrado_servidores_cadastro = df_filtrado_servidores_cadastro.join(
    df_contagem_desc_cargo.select("desc_cargo", "cod_cargo"),
    on="desc_cargo",
    how="left"
).withColumn(
    "cod_cargo",
    col("cod_cargo")
).select(*colunas_reordenadas)

display(df_filtrado_servidores_cadastro.limit(5))

desc_cargo,cod_cargo
AAD - AUXILIAR DE CONTABILIDADE,1
AAD - AUXILIAR DE RECURSOS HUMANOS,2
AAD - CONFERENTE,3
AAD-AUX DE RECURSOS MATERIAIS,4
AAD-AUXILIAR ADMINISTRATIVO,5


id,nome,cod_cargo,desc_cargo,cod_atividade,desc_atividade,cod_org_lotacao,org_lotacao,cod_orgsup_lotacao,orgsup_lotacao,situacao_vinculo,dt_inicio_afastamento,dt_fim_afastamento,uf_exercicio
2903139,AARAO FERREIRA LIMA NETO,null,null,0099,DIRETOR(A),26239,Universidade Federal do Pará,15000,Ministério da Educação,ATIVO PERMANENTE,null,null,PA
3137242,AARAO PEREIRA DE ARAUJO JUNIOR,1683,PROFESSOR ENS BASICO TECN TECNOLOGICO,null,null,26417,Instituto Federal da Paraíba,15000,Ministério da Educação,ATIVO PERMANENTE,null,null,PB
3174964,AARAO CARLOS LUZ MACAMBIRA,901,BIBLIOTECARIO-DOCUMENTALISTA,null,null,26405,Instituto Federal do Ceará,15000,Ministério da Educação,ATIVO PERMANENTE,null,null,CE
2868174,AARAO MEIR SERRUYA,1237,FISIOTERAPEUTA - 30H,null,null,26443,Empresa Brasileira de Serviços Hospitalares,15000,Ministério da Educação,CELETISTA/EMPREGADO,null,null,null
2903139,AARAO FERREIRA LIMA NETO,1682,PROFESSOR DO MAGISTERIO SUPERIOR,null,null,26239,Universidade Federal do Pará,15000,Ministério da Educação,ATIVO PERMANENTE,null,null,null


### • Inclusão da coluna de código para o vínculo do servidor
Processo de criação do identificador único para cada situação de vínculo, permitindo segmentação e análise dos diferentes tipos de vínculos dos servidores.

In [0]:
from pyspark.sql.functions import col, dense_rank
from pyspark.sql.window import Window

# Cria janela ordenada por situacao_vinculo
window_spec = Window.orderBy(col("situacao_vinculo").asc())

# Gera código sequencial cod_situacao_vinculo para cada valor distinto e não nulo de situacao_vinculo
df_cod_situacao_vinculo = (
    df_filtrado_servidores_cadastro
    .filter(col("situacao_vinculo").isNotNull())
    .select("situacao_vinculo")
    .distinct()
    .withColumn(
        "cod_situacao_vinculo",
        dense_rank().over(window_spec)
    )
    .orderBy(col("situacao_vinculo").asc())
)

display(df_cod_situacao_vinculo.limit(5))

# Realiza o join para inserir cod_situacao_vinculo e reordena as colunas para que cod_situacao_vinculo fique antes de situacao_vinculo
colunas_originais = df_filtrado_servidores_cadastro.columns
colunas_sem_cod = [col for col in colunas_originais if col != "cod_situacao_vinculo"]
pos_situacao = colunas_sem_cod.index("situacao_vinculo")
# Garante que cod_cargo permaneça nas colunas reordenadas
colunas_reordenadas = (
    colunas_sem_cod[:pos_situacao] +
    ["cod_situacao_vinculo"] +
    colunas_sem_cod[pos_situacao:]
)
if "cod_cargo" not in colunas_reordenadas and "cod_cargo" in df_filtrado_servidores_cadastro.columns:
    colunas_reordenadas.append("cod_cargo")

df_filtrado_servidores_cadastro = df_filtrado_servidores_cadastro.join(
    df_cod_situacao_vinculo.select("situacao_vinculo", "cod_situacao_vinculo"),
    on="situacao_vinculo",
    how="left"
).withColumn(
    "cod_situacao_vinculo",
    col("cod_situacao_vinculo")
).select(*colunas_reordenadas)

display(df_filtrado_servidores_cadastro.limit(5))

situacao_vinculo,cod_situacao_vinculo
ANISTIADO ADCT CF,1
APOS. COMPLEM. VIFER,2
ATIVO - DEC. JUDIC,3
ATIVO EM OUTRO ORGAO,4
ATIVO PERMANENTE,5


id,nome,cod_cargo,desc_cargo,cod_atividade,desc_atividade,cod_org_lotacao,org_lotacao,cod_orgsup_lotacao,orgsup_lotacao,cod_situacao_vinculo,situacao_vinculo,dt_inicio_afastamento,dt_fim_afastamento,uf_exercicio
2903139,AARAO FERREIRA LIMA NETO,null,null,0099,DIRETOR(A),26239,Universidade Federal do Pará,15000,Ministério da Educação,5,ATIVO PERMANENTE,null,null,PA
3137242,AARAO PEREIRA DE ARAUJO JUNIOR,1683,PROFESSOR ENS BASICO TECN TECNOLOGICO,null,null,26417,Instituto Federal da Paraíba,15000,Ministério da Educação,5,ATIVO PERMANENTE,null,null,PB
3174964,AARAO CARLOS LUZ MACAMBIRA,901,BIBLIOTECARIO-DOCUMENTALISTA,null,null,26405,Instituto Federal do Ceará,15000,Ministério da Educação,5,ATIVO PERMANENTE,null,null,CE
2868174,AARAO MEIR SERRUYA,1237,FISIOTERAPEUTA - 30H,null,null,26443,Empresa Brasileira de Serviços Hospitalares,15000,Ministério da Educação,14,CELETISTA/EMPREGADO,null,null,null
2903139,AARAO FERREIRA LIMA NETO,1682,PROFESSOR DO MAGISTERIO SUPERIOR,null,null,26239,Universidade Federal do Pará,15000,Ministério da Educação,5,ATIVO PERMANENTE,null,null,null


### • Tratamento de valores nulos nos campos de código e conversão das colunas de data
Processo de padronização dos campos de código para string e conversão das datas para o formato yyyy-MM-dd, mantendo nulos quando não há registro.

In [0]:
from pyspark.sql.functions import col, when, to_date

# Lista de campos de código para conversão e tratamento de nulos
colunas_cod = [
    "cod_atividade", "cod_org_lotacao", "cod_orgsup_lotacao", "cod_tipo_vinculo", "cod_cargo"
]
for coluna in colunas_cod:
    if coluna in df_filtrado_servidores_cadastro.columns:
        df_filtrado_servidores_cadastro = df_filtrado_servidores_cadastro.withColumn(
            coluna,
            when(col(coluna).isNull(), "0").otherwise(col(coluna).cast("string"))
        )

# Lista de campos de data para conversão
colunas_data = ["dt_inicio_afastamento", "dt_fim_afastamento"]
for coluna in colunas_data:
    if coluna in df_filtrado_servidores_cadastro.columns:
        df_filtrado_servidores_cadastro = df_filtrado_servidores_cadastro.withColumn(
            coluna,
            to_date(col(coluna), "yyyy-MM-dd")
        )

# Exibe resultado para conferência
display(df_filtrado_servidores_cadastro.limit(5))

id,nome,cod_cargo,desc_cargo,cod_atividade,desc_atividade,cod_org_lotacao,org_lotacao,cod_orgsup_lotacao,orgsup_lotacao,cod_situacao_vinculo,situacao_vinculo,dt_inicio_afastamento,dt_fim_afastamento,uf_exercicio
2903139,AARAO FERREIRA LIMA NETO,0,null,0099,DIRETOR(A),26239,Universidade Federal do Pará,15000,Ministério da Educação,5,ATIVO PERMANENTE,null,null,PA
3137242,AARAO PEREIRA DE ARAUJO JUNIOR,1683,PROFESSOR ENS BASICO TECN TECNOLOGICO,0,null,26417,Instituto Federal da Paraíba,15000,Ministério da Educação,5,ATIVO PERMANENTE,null,null,PB
3174964,AARAO CARLOS LUZ MACAMBIRA,901,BIBLIOTECARIO-DOCUMENTALISTA,0,null,26405,Instituto Federal do Ceará,15000,Ministério da Educação,5,ATIVO PERMANENTE,null,null,CE
2868174,AARAO MEIR SERRUYA,1237,FISIOTERAPEUTA - 30H,0,null,26443,Empresa Brasileira de Serviços Hospitalares,15000,Ministério da Educação,14,CELETISTA/EMPREGADO,null,null,null
2903139,AARAO FERREIRA LIMA NETO,1682,PROFESSOR DO MAGISTERIO SUPERIOR,0,null,26239,Universidade Federal do Pará,15000,Ministério da Educação,5,ATIVO PERMANENTE,null,null,null


### • Validação dos dados
Processo de validação de unicidade, consistência entre códigos e descrições, e garantia de que cada registro representa um servidor ativo único.

In [0]:
# Validação de unicidade de ID com nome
from pyspark.sql.functions import col, countDistinct

duplicados_id_nome = (
    df_filtrado_servidores_cadastro
    .groupBy("id")
    .agg(countDistinct("nome").alias("nomes_distintos"))
    .filter(col("nomes_distintos") > 1)
)
display(duplicados_id_nome)

# Checagem de consistência entre códigos e descrições
colunas_cod_desc = [
    ("cod_atividade", "desc_atividade"),
    ("cod_org_lotacao", "org_lotacao"),
    ("cod_orgsup_lotacao", "orgsup_lotacao"),
    ("cod_cargo", "desc_cargo"),
    ("cod_tipo_vinculo", "tipo_vinculo"),
    ("cod_situacao_vinculo", "situacao_vinculo")
]

for cod_col, desc_col in colunas_cod_desc:
    if cod_col in df_filtrado_servidores_cadastro.columns and desc_col in df_filtrado_servidores_cadastro.columns:
        inconsistentes = (
            df_filtrado_servidores_cadastro
            .groupBy(cod_col)
            .agg(countDistinct(desc_col).alias("qtd_desc"))
            .filter(col("qtd_desc") > 1)
        )
        display(inconsistentes)

id,nomes_distintos


cod_atividade,qtd_desc


cod_org_lotacao,qtd_desc


cod_orgsup_lotacao,qtd_desc


cod_cargo,qtd_desc


cod_situacao_vinculo,qtd_desc


### • Sobreescrevendo as tabela com os dados tratado

In [0]:

df_filtrado_servidores_cadastro.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("servidores.silver.servidores_cadastro")

### • Documentação e comentários no catálogo de dados
Processo de geração de catálogo com comentários para as tabelas e colunas, detalhando o significado de cada atributo.

In [0]:
%sql
-- Comentário para a tabela servidores_cadastro
COMMENT ON TABLE servidores_cadastro IS 'Tabela de dados cadastrais dos servidores ativos, com informações limpas, padronizadas e prontas para análises de vínculos, cargos, atividades e órgãos de lotação.';

-- Comentários para o catálogo de dados da tabela servidores_cadastro
COMMENT ON COLUMN servidores_cadastro.id IS 'Identificador único do servidor';
COMMENT ON COLUMN servidores_cadastro.nome IS 'Nome completo do servidor';
COMMENT ON COLUMN servidores_cadastro.cod_cargo IS 'Código do cargo ocupado pelo servidor';
COMMENT ON COLUMN servidores_cadastro.desc_cargo IS 'Descrição do cargo ocupado pelo servidor';
COMMENT ON COLUMN servidores_cadastro.cod_atividade IS 'Código da atividade exercida pelo servidor';
COMMENT ON COLUMN servidores_cadastro.desc_atividade IS 'Descrição da atividade exercida pelo servidor';
COMMENT ON COLUMN servidores_cadastro.cod_org_lotacao IS 'Código do órgão de lotação do servidor';
COMMENT ON COLUMN servidores_cadastro.org_lotacao IS 'Nome do órgão de lotação do servidor';
COMMENT ON COLUMN servidores_cadastro.cod_orgsup_lotacao IS 'Código do órgão superior de lotação';
COMMENT ON COLUMN servidores_cadastro.orgsup_lotacao IS 'Nome do órgão superior de lotação';
COMMENT ON COLUMN servidores_cadastro.cod_situacao_vinculo IS 'Código da situação do vínculo do servidor';
COMMENT ON COLUMN servidores_cadastro.situacao_vinculo IS 'Situação atual do vínculo do servidor';
COMMENT ON COLUMN servidores_cadastro.dt_inicio_afastamento IS 'Data de início do afastamento do servidor, formato yyyy-MM-dd, NULL quando não aplicável';
COMMENT ON COLUMN servidores_cadastro.dt_fim_afastamento IS 'Data de término do afastamento do servidor, formato yyyy-MM-dd, NULL quando não aplicável';
COMMENT ON COLUMN servidores_cadastro.uf_exercicio IS 'Unidade Federativa de exercício do servidor';

DESCRIBE servidores_cadastro;

col_name,data_type,comment
id,string,Identificador único do servidor
nome,string,Nome completo do servidor
cod_cargo,string,Código do cargo ocupado pelo servidor
desc_cargo,string,Descrição do cargo ocupado pelo servidor
cod_atividade,string,Código da atividade exercida pelo servidor
desc_atividade,string,Descrição da atividade exercida pelo servidor
cod_org_lotacao,string,Código do órgão de lotação do servidor
org_lotacao,string,Nome do órgão de lotação do servidor
cod_orgsup_lotacao,string,Código do órgão superior de lotação
orgsup_lotacao,string,Nome do órgão superior de lotação


In [0]:
%sql
DESCRIBE DETAIL servidores_cadastro;


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,fb9e4e81-3021-4250-9b58-9314db4cedd5,servidores.silver.servidores_cadastro,"Tabela de dados cadastrais dos servidores ativos, com informações limpas, padronizadas e prontas para análises de vínculos, cargos, atividades e órgãos de lotação.",,2025-12-22T04:17:28.241Z,2025-12-22T04:18:23.000Z,List(),List(),1,13666882,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


### `aposentados_cadastro`

- **Seleção de colunas para análise e modelagem:**
    - `Id_SERVIDOR_PORTAL`: Identificador único do servidor, essencial para garantir unicidade e rastreabilidade dos registros.
    - `NOME`: Nome completo do servidor, necessário para identificação individual e validação de dados.
    - `DESCRICAO_CARGO`: Cargo ocupado, fundamental para análises de distribuição e remuneração por função.
    - `CODIGO_ATIVIDADE`, `ATIVIDADE`: Código e descrição da atividade, permitem segmentar servidores por área de atuação.
    - `COD_ORG_LOTACAO`, `ORG_LOTACAO`: Código e órgão de lotação, importantes para análises por unidade administrativa.
    - `COD_ORGSUP_LOTACAO`, `ORGSUP_LOTACAO`: Código e órgão superior, viabilizam agrupamentos por estrutura hierárquica.
    - `SITUACAO_VINCULO`: Situação do vínculo, permite identificar servidores ativos, afastados, licenciados, etc.
    - `DATA_INICIO_AFASTAMENTO`, `DATA_TERMINO_AFASTAMENTO`: Datas de afastamento, necessárias para análises de licenças e ausências.
    - `UF_EXERCICIO`: UF de exercício, relevante para segmentação geográfica dos servidores.
    - `cod_descricao`: Coluna criada para armazenar o código referente à descrição do cargo, garantindo integridade referencial e facilitando análises por função.
    - `cod_vinculo`: Coluna criada para armazenar o código referente ao vínculo do servidor, permitindo segmentação e análise dos diferentes tipos de vínculos.

- **Padronização e renomeação dos campos:**  
    - `Id_SERVIDOR_PORTAL` → `id`  
    - `NOME` → `nome`  
    - `DESCRICAO_CARGO` → `desc_cargo`  
    - `CODIGO_ATIVIDADE` → `cod_atividade`  
    - `ATIVIDADE` → `desc_atividade`  
    - `COD_ORG_LOTACAO` → `cod_org_lotacao`  
    - `ORG_LOTACAO` → `org_lotacao`  
    - `COD_ORGSUP_LOTACAO` → `cod_orgsup_lotacao`  
    - `ORGSUP_LOTACAO` → `orgsup_lotacao`  
    - `SITUACAO_VINCULO` → `situacao_vinculo`  
    - `DATA_INICIO_AFASTAMENTO` → `dt_inicio_afastamento`  
    - `DATA_TERMINO_AFASTAMENTO` → `dt_termino_afastamento`  
    - `UF_EXERCICIO` → `uf_exercicio`  
- Inclusão das colunas:
    - `cod_descricao`: código exclusivo para cada descrição de cargo, garantindo integridade referencial e facilitando análises agregadas por função.
    - `cod_vinculo`: código para cada tipo de vínculo, permitindo segmentação precisa dos servidores conforme vínculo funcional e aprimorando a rastreabilidade dos dados.

- **Definição dos campos obrigatórios e opcionais:**  
  - Os campos `id` e `nome` são obrigatórios e não podem conter valores nulos.
  - Os demais campos são opcionais e podem apresentar valores nulos, preservando a integridade e a representatividade dos dados originais.

- **Conversão e padronização de tipos:**  
    - Códigos convertidos para inteiro.  
    - Datas para formato `yyyy-MM-dd`.  
    - Demais campos mantidos como string.

- **Tratamento de valores nulos e ausentes:**  
    - Campos de datas de afastamento permanecem nulos, indicando ausência de registros no período.  
    - Outros campos podem ter nulos em casos de cadastro incompleto ou vínculos atípicos.
    - Campos `cod_descricao` e `cod_vinculo` mantêm nulos quando não há registro correspondente.

- **Validação dos dados:**  
    - Garantia de unicidade em `id`.  
    - Checagem de consistência entre códigos e descrições.  
    - Validação de integridade referencial entre códigos e suas respectivas descrições.
    - Verificação de ausência de duplicidades e registros inconsistentes.
    - Conferência de domínios esperados para atributos categóricos.

- **Sobreescrita da tabela tratada:**  
    - A tabela `servidores_cadastro` tratada é sobreescrita após o processo de limpeza, validação e padronização, garantindo que apenas registros válidos e consistentes sejam mantidos para análise.

- **Documentação e comentários no catálogo de dados:**  
    - Comentários foram adicionados para as colunas e para a tabela no catálogo de dados, detalhando o significado de cada atributo, incluindo as etapas de inclusão das colunas `cod_descricao` e `cod_vinculo`.

### • Seleção de colunas para análise e modelagem
Seleciona apenas as colunas essenciais para garantir unicidade, rastreabilidade e representatividade dos dados dos servidores aposentados.

In [0]:
%sql

select * from servidores.silver.aposentados_cadastro limit 5

Id_SERVIDOR_PORTAL,NOME,CPF,MATRICULA,COD_TIPO_APOSENTADORIA,TIPO_APOSENTADORIA,DATA_APOSENTADORIA,DESCRICAO_CARGO,COD_UORG_LOTACAO,UORG_LOTACAO,COD_ORG_LOTACAO,ORG_LOTACAO,COD_ORGSUP_LOTACAO,ORGSUP_LOTACAO,COD_TIPO_VINCULO,TIPO_VINCULO,SITUACAO_VINCULO,REGIME_JURIDICO,JORNADA_DE_TRABALHO,DATA_INGRESSO_CARGOFUNCAO,DATA_NOMEACAO_CARGOFUNCAO,DATA_INGRESSO_ORGAO,DOCUMENTO_INGRESSO_SERVICOPUBLICO,DATA_DIPLOMA_INGRESSO_SERVICOPUBLICO,DIPLOMA_INGRESSO_CARGOFUNCAO,DIPLOMA_INGRESSO_ORGAO,DIPLOMA_INGRESSO_SERVICOPUBLICO
2025221,AARAO DE ANDRADE LIMA,***.559.144-**,033****,01,APOSENTADORIA VOLUNTARIA,12/11/2012,PROFESSOR DO MAGISTERIO SUPERIOR,-3,Inválido,26252,Universidade Federal de Campina Grande - PB,15000,Ministério da Educação,5,Aposentadoria,APOSENTADO,REGIME JURIDICO UNICO,DEDICACAO EXCLUSIVA,01/03/2013,null,10/04/2002,SN,10/03/1977,null,LEI,PORTARIA
3594060,AARAO MOREIRA DA SILVA,***.924.486-**,048****,01,APOSENTADORIA VOLUNTARIA,16/05/2011,AGENTE DE SAUDE PUBLICA,-3,Inválido,25000,Ministério da Saúde,-1,Sem informação,5,Aposentadoria,APOSENTADO,REGIME JURIDICO UNICO,40 HORAS SEMANAIS,01/03/2006,null,29/06/2010,SN,06/10/1975,null,PORTARIA,CONTRATO
87244,ABA ISRAEL COHEN PERSIANO,***.681.016-**,032****,01,APOSENTADORIA VOLUNTARIA,05/05/2014,PROFESSOR DO MAGISTERIO SUPERIOR,-3,Inválido,26238,Universidade Federal de Minas Gerais,15000,Ministério da Educação,5,Aposentadoria,APOSENTADO,REGIME JURIDICO UNICO,DEDICACAO EXCLUSIVA,29/01/1997,null,29/03/1978,0000000SN,18/04/1974,null,PORTARIA,PORTARIA
661149,ABADIA APARECIDA FAUSTINO,***.430.802-**,108****,01,APOSENTADORIA VOLUNTARIA,19/12/2024,AUXILIAR OPERACIONAL SERV DIVERSOS - NA,-3,Inválido,40806,DEP.DE CENTRAL.SERV.DE INATIVOS E PENS.,17500,MIN GESTAO E INOV EM SERV PUBLICOS,5,Aposentadoria,APOSENTADO,REGIME JURIDICO UNICO,40 HORAS SEMANAIS,01/08/1984,null,19/12/2024,NI,01/08/1984,null,PORTARIA,CONTRATO
3336973,ABADIA BELCHIOR GOMES,***.078.406-**,041****,01,APOSENTADORIA VOLUNTARIA,30/11/2012,SERVENTE DE LIMPEZA,-3,Inválido,26274,Fundação Universidade Federal Uberlândia,15000,Ministério da Educação,5,Aposentadoria,APOSENTADO,REGIME JURIDICO UNICO,40 HORAS SEMANAIS,01/03/2005,null,01/02/1982,000000S/N,01/02/1982,null,CONTRATO,CONTRATO


In [0]:
df_aposentados_cadastro = spark.table("servidores.silver.aposentados_cadastro")

display(df_aposentados_cadastro.limit)

<bound method DataFrame.limit of DataFrame[Id_SERVIDOR_PORTAL: string, NOME: string, CPF: string, MATRICULA: string, COD_TIPO_APOSENTADORIA: string, TIPO_APOSENTADORIA: string, DATA_APOSENTADORIA: string, DESCRICAO_CARGO: string, COD_UORG_LOTACAO: string, UORG_LOTACAO: string, COD_ORG_LOTACAO: string, ORG_LOTACAO: string, COD_ORGSUP_LOTACAO: string, ORGSUP_LOTACAO: string, COD_TIPO_VINCULO: string, TIPO_VINCULO: string, SITUACAO_VINCULO: string, REGIME_JURIDICO: string, JORNADA_DE_TRABALHO: string, DATA_INGRESSO_CARGOFUNCAO: string, DATA_NOMEACAO_CARGOFUNCAO: string, DATA_INGRESSO_ORGAO: string, DOCUMENTO_INGRESSO_SERVICOPUBLICO: string, DATA_DIPLOMA_INGRESSO_SERVICOPUBLICO: string, DIPLOMA_INGRESSO_CARGOFUNCAO: string, DIPLOMA_INGRESSO_ORGAO: string, DIPLOMA_INGRESSO_SERVICOPUBLICO: string]>

In [0]:
colunas_selecionadas = [
    "Id_SERVIDOR_PORTAL", "NOME", "DESCRICAO_CARGO", "COD_TIPO_APOSENTADORIA", "TIPO_APOSENTADORIA", "DATA_APOSENTADORIA", "COD_ORG_LOTACAO", "ORG_LOTACAO", "COD_ORGSUP_LOTACAO", "ORGSUP_LOTACAO"
]
df_aposentados_cadastro = df_aposentados_cadastro.select(*colunas_selecionadas)
display(df_aposentados_cadastro.limit(5))

Id_SERVIDOR_PORTAL,NOME,DESCRICAO_CARGO,COD_TIPO_APOSENTADORIA,TIPO_APOSENTADORIA,DATA_APOSENTADORIA,COD_ORG_LOTACAO,ORG_LOTACAO,COD_ORGSUP_LOTACAO,ORGSUP_LOTACAO
2025221,AARAO DE ANDRADE LIMA,PROFESSOR DO MAGISTERIO SUPERIOR,01,APOSENTADORIA VOLUNTARIA,12/11/2012,26252,Universidade Federal de Campina Grande - PB,15000,Ministério da Educação
3594060,AARAO MOREIRA DA SILVA,AGENTE DE SAUDE PUBLICA,01,APOSENTADORIA VOLUNTARIA,16/05/2011,25000,Ministério da Saúde,-1,Sem informação
87244,ABA ISRAEL COHEN PERSIANO,PROFESSOR DO MAGISTERIO SUPERIOR,01,APOSENTADORIA VOLUNTARIA,05/05/2014,26238,Universidade Federal de Minas Gerais,15000,Ministério da Educação
661149,ABADIA APARECIDA FAUSTINO,AUXILIAR OPERACIONAL SERV DIVERSOS - NA,01,APOSENTADORIA VOLUNTARIA,19/12/2024,40806,DEP.DE CENTRAL.SERV.DE INATIVOS E PENS.,17500,MIN GESTAO E INOV EM SERV PUBLICOS
3336973,ABADIA BELCHIOR GOMES,SERVENTE DE LIMPEZA,01,APOSENTADORIA VOLUNTARIA,30/11/2012,26274,Fundação Universidade Federal Uberlândia,15000,Ministério da Educação


### • Padronização e renomeação dos campos
Renomeia as colunas para o padrão sintético definido na documentação.

In [0]:
mapeamento_colunas = {
    "Id_SERVIDOR_PORTAL": "id",
    "NOME": "nome",
    "DESCRICAO_CARGO": "desc_cargo",
    "COD_TIPO_APOSENTADORIA": "cod_tipo_aposentadoria",
    "TIPO_APOSENTADORIA": "tipo_aposentadoria",
    "DATA_APOSENTADORIA": "dt_aposentadoria",
    "COD_ORG_LOTACAO": "cod_org_lotacao",
    "ORG_LOTACAO": "org_lotacao",
    "COD_ORGSUP_LOTACAO": "cod_orgsup_lotacao",
    "ORGSUP_LOTACAO": "orgsup_lotacao",
}
for original, novo in mapeamento_colunas.items():
    if original in df_aposentados_cadastro.columns:
        df_aposentados_cadastro = df_aposentados_cadastro.withColumnRenamed(original, novo)
display(df_aposentados_cadastro.limit(5))

id,nome,desc_cargo,cod_tipo_aposentadoria,tipo_aposentadoria,dt_aposentadoria,cod_org_lotacao,org_lotacao,cod_orgsup_lotacao,orgsup_lotacao
2025221,AARAO DE ANDRADE LIMA,PROFESSOR DO MAGISTERIO SUPERIOR,01,APOSENTADORIA VOLUNTARIA,12/11/2012,26252,Universidade Federal de Campina Grande - PB,15000,Ministério da Educação
3594060,AARAO MOREIRA DA SILVA,AGENTE DE SAUDE PUBLICA,01,APOSENTADORIA VOLUNTARIA,16/05/2011,25000,Ministério da Saúde,-1,Sem informação
87244,ABA ISRAEL COHEN PERSIANO,PROFESSOR DO MAGISTERIO SUPERIOR,01,APOSENTADORIA VOLUNTARIA,05/05/2014,26238,Universidade Federal de Minas Gerais,15000,Ministério da Educação
661149,ABADIA APARECIDA FAUSTINO,AUXILIAR OPERACIONAL SERV DIVERSOS - NA,01,APOSENTADORIA VOLUNTARIA,19/12/2024,40806,DEP.DE CENTRAL.SERV.DE INATIVOS E PENS.,17500,MIN GESTAO E INOV EM SERV PUBLICOS
3336973,ABADIA BELCHIOR GOMES,SERVENTE DE LIMPEZA,01,APOSENTADORIA VOLUNTARIA,30/11/2012,26274,Fundação Universidade Federal Uberlândia,15000,Ministério da Educação


### • Definição dos campos obrigatórios e opcionais
Define id e nome como obrigatórios, demais campos como opcionais.

### • Conversão e padronização de tipos
Converte campos de datas para yyyy-MM-dd.

In [0]:

for coluna in colunas_cod:
    if coluna in df_aposentados_cadastro.columns:
        df_aposentados_cadastro = df_aposentados_cadastro.withColumn(
            coluna,
            when(col(coluna).isNull(), "0").otherwise(col(coluna).cast("string"))
        )
if "dt_aposentadoria" in df_aposentados_cadastro.columns:
    df_aposentados_cadastro = df_aposentados_cadastro.withColumn(
        "dt_aposentadoria",
        try_to_date(col("dt_aposentadoria"), "dd/MM/yyyy")
    )
display(df_aposentados_cadastro.limit(5))

id,nome,desc_cargo,cod_tipo_aposentadoria,tipo_aposentadoria,dt_aposentadoria,cod_org_lotacao,org_lotacao,cod_orgsup_lotacao,orgsup_lotacao
2025221,AARAO DE ANDRADE LIMA,PROFESSOR DO MAGISTERIO SUPERIOR,01,APOSENTADORIA VOLUNTARIA,2012-11-12,26252,Universidade Federal de Campina Grande - PB,15000,Ministério da Educação
3594060,AARAO MOREIRA DA SILVA,AGENTE DE SAUDE PUBLICA,01,APOSENTADORIA VOLUNTARIA,2011-05-16,25000,Ministério da Saúde,-1,Sem informação
87244,ABA ISRAEL COHEN PERSIANO,PROFESSOR DO MAGISTERIO SUPERIOR,01,APOSENTADORIA VOLUNTARIA,2014-05-05,26238,Universidade Federal de Minas Gerais,15000,Ministério da Educação
661149,ABADIA APARECIDA FAUSTINO,AUXILIAR OPERACIONAL SERV DIVERSOS - NA,01,APOSENTADORIA VOLUNTARIA,2024-12-19,40806,DEP.DE CENTRAL.SERV.DE INATIVOS E PENS.,17500,MIN GESTAO E INOV EM SERV PUBLICOS
3336973,ABADIA BELCHIOR GOMES,SERVENTE DE LIMPEZA,01,APOSENTADORIA VOLUNTARIA,2012-11-30,26274,Fundação Universidade Federal Uberlândia,15000,Ministério da Educação


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank, col

# Gera código sequencial cod_cargo para cada valor distinto e não nulo de desc_cargo
window_cargo = Window.orderBy(col("desc_cargo").asc())
df_cod_cargo = (
    df_aposentados_cadastro
    .filter(col("desc_cargo").isNotNull())
    .select("desc_cargo")
    .distinct()
    .withColumn("cod_cargo", dense_rank().over(window_cargo))
)

# Realiza o join para inserir cod_cargo
if "desc_cargo" in df_aposentados_cadastro.columns:
    df_aposentados_cadastro = df_aposentados_cadastro.join(
        df_cod_cargo, on="desc_cargo", how="left"
    )

# Reordena as colunas para que id, nome, cod_cargo, desc_cargo fiquem no início
colunas = df_aposentados_cadastro.columns
colunas_inicio = [c for c in ["id", "nome", "cod_cargo", "desc_cargo"] if c in colunas]
colunas_restantes = [c for c in colunas if c not in colunas_inicio]
df_aposentados_cadastro = df_aposentados_cadastro.select(*colunas_inicio, *colunas_restantes)

display(df_aposentados_cadastro.limit(5))

id,nome,cod_cargo,desc_cargo,cod_tipo_aposentadoria,tipo_aposentadoria,dt_aposentadoria,cod_org_lotacao,org_lotacao,cod_orgsup_lotacao,orgsup_lotacao
2025221,AARAO DE ANDRADE LIMA,1028,PROFESSOR DO MAGISTERIO SUPERIOR,01,APOSENTADORIA VOLUNTARIA,2012-11-12,26252,Universidade Federal de Campina Grande - PB,15000,Ministério da Educação
3594060,AARAO MOREIRA DA SILVA,86,AGENTE DE SAUDE PUBLICA,01,APOSENTADORIA VOLUNTARIA,2011-05-16,25000,Ministério da Saúde,-1,Sem informação
87244,ABA ISRAEL COHEN PERSIANO,1028,PROFESSOR DO MAGISTERIO SUPERIOR,01,APOSENTADORIA VOLUNTARIA,2014-05-05,26238,Universidade Federal de Minas Gerais,15000,Ministério da Educação
661149,ABADIA APARECIDA FAUSTINO,522,AUXILIAR OPERACIONAL SERV DIVERSOS - NA,01,APOSENTADORIA VOLUNTARIA,2024-12-19,40806,DEP.DE CENTRAL.SERV.DE INATIVOS E PENS.,17500,MIN GESTAO E INOV EM SERV PUBLICOS
3336973,ABADIA BELCHIOR GOMES,1112,SERVENTE DE LIMPEZA,01,APOSENTADORIA VOLUNTARIA,2012-11-30,26274,Fundação Universidade Federal Uberlândia,15000,Ministério da Educação


### • Tratamento de valores nulos e ausentes
Substitui valores inválidos ou ausentes por None.

In [0]:
valores_invalidos = ["-1", "Sem informaç", "Inválido", "Sem informação", ""]
colunas_tratar = ["desc_cargo", "orgsup_lotacao", "tipo_aposentadoria"]

from pyspark.sql.functions import col

for coluna in colunas_tratar:
    if coluna in df_aposentados_cadastro.columns:
        df_aposentados_cadastro = df_aposentados_cadastro.replace(valores_invalidos, None, subset=[coluna])

display(df_aposentados_cadastro.limit(5))

id,nome,cod_cargo,desc_cargo,cod_tipo_aposentadoria,tipo_aposentadoria,dt_aposentadoria,cod_org_lotacao,org_lotacao,cod_orgsup_lotacao,orgsup_lotacao
2025221,AARAO DE ANDRADE LIMA,1028,PROFESSOR DO MAGISTERIO SUPERIOR,01,APOSENTADORIA VOLUNTARIA,2012-11-12,26252,Universidade Federal de Campina Grande - PB,15000,Ministério da Educação
3594060,AARAO MOREIRA DA SILVA,86,AGENTE DE SAUDE PUBLICA,01,APOSENTADORIA VOLUNTARIA,2011-05-16,25000,Ministério da Saúde,-1,null
87244,ABA ISRAEL COHEN PERSIANO,1028,PROFESSOR DO MAGISTERIO SUPERIOR,01,APOSENTADORIA VOLUNTARIA,2014-05-05,26238,Universidade Federal de Minas Gerais,15000,Ministério da Educação
661149,ABADIA APARECIDA FAUSTINO,522,AUXILIAR OPERACIONAL SERV DIVERSOS - NA,01,APOSENTADORIA VOLUNTARIA,2024-12-19,40806,DEP.DE CENTRAL.SERV.DE INATIVOS E PENS.,17500,MIN GESTAO E INOV EM SERV PUBLICOS
3336973,ABADIA BELCHIOR GOMES,1112,SERVENTE DE LIMPEZA,01,APOSENTADORIA VOLUNTARIA,2012-11-30,26274,Fundação Universidade Federal Uberlândia,15000,Ministério da Educação


In [0]:
from pyspark.sql.functions import col, when

# Lista de campos de código para tratamento
colunas_cod = [
    "cod_tipo_aposentadoria", "cod_org_lotacao", "cod_orgsup_lotacao", "cod_tipo_vinculo", "cod_cargo"
]

for coluna in colunas_cod:
    if coluna in df_aposentados_cadastro.columns:
        df_aposentados_cadastro = df_aposentados_cadastro.withColumn(
            coluna,
            when(col(coluna).isNull() | (col(coluna).cast("int") < 0), "0").otherwise(col(coluna).cast("string"))
        )

display(df_aposentados_cadastro.limit(5))

id,nome,cod_cargo,desc_cargo,cod_tipo_aposentadoria,tipo_aposentadoria,dt_aposentadoria,cod_org_lotacao,org_lotacao,cod_orgsup_lotacao,orgsup_lotacao
2025221,AARAO DE ANDRADE LIMA,1028,PROFESSOR DO MAGISTERIO SUPERIOR,01,APOSENTADORIA VOLUNTARIA,2012-11-12,26252,Universidade Federal de Campina Grande - PB,15000,Ministério da Educação
3594060,AARAO MOREIRA DA SILVA,86,AGENTE DE SAUDE PUBLICA,01,APOSENTADORIA VOLUNTARIA,2011-05-16,25000,Ministério da Saúde,0,null
87244,ABA ISRAEL COHEN PERSIANO,1028,PROFESSOR DO MAGISTERIO SUPERIOR,01,APOSENTADORIA VOLUNTARIA,2014-05-05,26238,Universidade Federal de Minas Gerais,15000,Ministério da Educação
661149,ABADIA APARECIDA FAUSTINO,522,AUXILIAR OPERACIONAL SERV DIVERSOS - NA,01,APOSENTADORIA VOLUNTARIA,2024-12-19,40806,DEP.DE CENTRAL.SERV.DE INATIVOS E PENS.,17500,MIN GESTAO E INOV EM SERV PUBLICOS
3336973,ABADIA BELCHIOR GOMES,1112,SERVENTE DE LIMPEZA,01,APOSENTADORIA VOLUNTARIA,2012-11-30,26274,Fundação Universidade Federal Uberlândia,15000,Ministério da Educação


### • Validação dos dados
Validação de unicidade, consistência entre códigos e descrições.

In [0]:
from pyspark.sql.functions import col, countDistinct

# Validação de unicidade de id com nome
duplicados_id_nome = (
    df_aposentados_cadastro
    .groupBy("id")
    .agg(countDistinct("nome").alias("nomes_distintos"))
    .filter(col("nomes_distintos") > 1)
)
display(duplicados_id_nome)

# Validação de consistência entre códigos e descrições
colunas_cod_desc = [
    ("cod_tipo_aposentadoria", "tipo_aposentadoria"),
    ("cod_org_lotacao", "org_lotacao"),
    ("cod_orgsup_lotacao", "orgsup_lotacao"),
    ("cod_cargo", "desc_cargo"),
]

for cod_col, desc_col in colunas_cod_desc:
    if cod_col in df_aposentados_cadastro.columns and desc_col in df_aposentados_cadastro.columns:
        inconsistentes = (
            df_aposentados_cadastro
            .groupBy(cod_col)
            .agg(countDistinct(desc_col).alias("qtd_desc"))
            .filter(col("qtd_desc") > 1)
        )
        display(inconsistentes)

id,nomes_distintos


cod_tipo_aposentadoria,qtd_desc


cod_org_lotacao,qtd_desc


cod_orgsup_lotacao,qtd_desc


cod_cargo,qtd_desc


### • Sobreescrita da tabela tratada
A tabela tratada é sobreescrita após limpeza, validação e padronização.

In [0]:
df_aposentados_cadastro.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("servidores.silver.aposentados_cadastro")

### • Documentação e comentários no catálogo de dados

In [0]:
%sql
-- Processo de geração de catálogo com comentários para as tabelas e colunas, detalhando o significado de cada atributo.
COMMENT ON TABLE servidores.silver.aposentados_cadastro IS 'Tabela de dados cadastrais dos servidores aposentados, com informações limpas, padronizadas e prontas para análises de vínculos, cargos, tipos de aposentadoria e órgãos de lotação.';
COMMENT ON COLUMN servidores.silver.aposentados_cadastro.id IS 'Identificador único do servidor aposentado';
COMMENT ON COLUMN servidores.silver.aposentados_cadastro.nome IS 'Nome completo do servidor aposentado';
COMMENT ON COLUMN servidores.silver.aposentados_cadastro.cod_cargo IS 'Código do cargo ocupado pelo servidor antes da aposentadoria';
COMMENT ON COLUMN servidores.silver.aposentados_cadastro.desc_cargo IS 'Descrição do cargo ocupado pelo servidor antes da aposentadoria';
COMMENT ON COLUMN servidores.silver.aposentados_cadastro.cod_tipo_aposentadoria IS 'Código do tipo de aposentadoria';
COMMENT ON COLUMN servidores.silver.aposentados_cadastro.tipo_aposentadoria IS 'Descrição do tipo de aposentadoria';
COMMENT ON COLUMN servidores.silver.aposentados_cadastro.dt_aposentadoria IS 'Data da aposentadoria, formato yyyy-MM-dd';
COMMENT ON COLUMN servidores.silver.aposentados_cadastro.cod_org_lotacao IS 'Código do órgão de lotação do servidor';
COMMENT ON COLUMN servidores.silver.aposentados_cadastro.org_lotacao IS 'Nome do órgão de lotação do servidor';
COMMENT ON COLUMN servidores.silver.aposentados_cadastro.cod_orgsup_lotacao IS 'Código do órgão superior de lotação';
COMMENT ON COLUMN servidores.silver.aposentados_cadastro.orgsup_lotacao IS 'Nome do órgão superior de lotação';

In [0]:
%sql
DESCRIBE DETAIL aposentados_cadastro;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,8b720570-f37c-41be-82bd-fbc3d32c0014,servidores.silver.aposentados_cadastro,"Tabela de dados cadastrais dos servidores aposentados, com informações limpas, padronizadas e prontas para análises de vínculos, cargos, tipos de aposentadoria e órgãos de lotação.",,2025-12-22T04:17:36.225Z,2025-12-22T04:18:49.000Z,List(),List(),1,7544981,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false



### 3 - Processos de análise, filtragem e transformação das colunas das tabelas `servidores_remuneracao` e 'aposentados_remuneracao'


### `servidores_remuneracao`

- **Seleção de colunas para análise e modelagem:**
    - `ano`, `mes`, `id`, `nome`, `remuneracao_bruta_brl`, `abate_teto_brl`, `gratificacao_natalina_brl`, `ferias_brl`, `remuneracoes_eventuais_brl`, `irrf_brl`, `pss_rpgs_brl`, `demais_deducoes_brl`, `taxa_ocupacao_imovel_funcional_brl`, `remuneracao_apos_deducoes_obrigatorias_brl`, `indenizacao_registradas_sistemas_pessoal_civil_brl`, `indenizacao_programa_desligamento_voluntario_mp_792_2017_brl`, `total_indenizacao_brl`.
    - Essas colunas representam as principais medidas e dimensões para análise financeira e administrativa dos servidores.

- **Padronização e renomeação dos campos:**
    - Colunas originais foram renomeadas para o padrão sintético, facilitando consultas, joins e padronização entre tabelas.
    - Exemplo: `REMUNERACAO_BASICA_BRUTA_BRL` → `remuneracao_bruta_brl`, `FERIAS_BRL` → `ferias_brl`, etc.

- **Definição dos campos obrigatórios e opcionais:**
    - Os campos `ano`, `mes`, `id`, `remuneracao_bruta_brl` são obrigatórios para garantir a identificação única e a integridade dos registros.
    - Os demais campos são opcionais e podem conter valores nulos, preservando a representatividade dos dados originais.

- **Conversão e padronização de tipos:**
    - Valores financeiros convertidos para tipo Double, facilitando cálculos e agregações.
    - Códigos e identificadores convertidos para string.
    - Datas mantidas como string (`ano`, `mes`).

- **Tratamento de valores nulos e ausentes:**
    - Campos financeiros e opcionais podem permanecer nulos, indicando ausência de registro ou informação não aplicável.
    - Valores nulos em campos obrigatórios são tratados e validados.

- **Validação dos dados:**
    - Garantia de unicidade em `id`, `ano`, `mes`.
    - Checagem de consistência entre identificadores e nomes.
    - Validação de integridade referencial entre `id` e cadastro de servidores.
    - Verificação de ausência de duplicidades e registros inconsistentes.
    - Conferência de domínios esperados para atributos categóricos e financeiros.

- **Sobreescrita da tabela tratada:**
    - A tabela `servidores_remuneracao` tratada é sobreescrita após o processo de limpeza, validação e padronização, garantindo que apenas registros válidos e consistentes sejam mantidos para análise.

- **Documentação e comentários no catálogo de dados:**
    - Comentários detalhados foram adicionados para as colunas e para a tabela no catálogo de dados, explicando o significado, uso e regras de negócio de cada atributo.

In [0]:
%sql
select * from servidores.silver.servidores_remuneracao limit 5

ANO,MES,Id_SERVIDOR_PORTAL,CPF,NOME,REMUNERACAO_BASICA_BRUTA_BRL,REMUNERACAO_BASICA_BRUTA_USD,ABATE-TETO_BRL,ABATE-TETO_USD,GRATIFICACAO_NATALINA_BRL,GRATIFICACAO_NATALINA_USD,ABATE-TETO_DA_GRATIFICACAO_NATALINA_BRL,ABATE-TETO_DA_GRATIFICACAO_NATALINA_USD,FERIAS_BRL,FERIAS_USD,OUTRAS_REMUNERACOES_EVENTUAIS_BRL,OUTRAS_REMUNERACOES_EVENTUAIS_USD,IRRF_BRL,IRRF_USD,PSS/RPGS_BRL,PSS/RPGS_USD,DEMAIS_DEDUCOES_BRL,DEMAIS_DEDUCOES_USD,PENSAO_MILITAR_BRL,PENSAO_MILITAR_USD,FUNDO_DE_SAUDE_BRL,FUNDO_DE_SAUDE_USD,TAXA_DE_OCUPACAO_IMOVEL_FUNCIONAL_BRL,TAXA_DE_OCUPACAO_IMOVEL_FUNCIONAL_USD,REMUNERACAO_APOS_DEDUCOES_OBRIGATORIAS_BRL,REMUNERACAO_APOS_DEDUCOES_OBRIGATORIAS_USD,INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_CIVIL_BRL,INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_CIVIL_USD,INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_MILITAR_BRL,INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_MILITAR_USD,INDENIZACAO_PROGRAMA_DESLIGAMENTO_VOLUNTARIO_MP_792/2017_BRL,INDENIZACAO_PROGRAMA_DESLIGAMENTO_VOLUNTARIO_MP_792/2017_USD,TOTAL_DE_INDENIZACAO_BRL,TOTAL_DE_INDENIZACAO_USD
2025,10,3174964,***.017.623-**,AARAO CARLOS LUZ MACAMBIRA,"12577,90","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","-2112,23","0,00","-1592,59","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","8873,08","0,00","1294,96","0,00","0,00","0,00","0,00","0,00","1294,96","0,00"
2025,10,2903139,***.116.132-**,AARAO FERREIRA LIMA NETO,"28342,89","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","803,52","0,00","-6267,89","0,00","-3049,58","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","19828,94","0,00","1000,00","0,00","0,00","0,00","0,00","0,00","1000,00","0,00"
2025,10,3137242,***.031.184-**,AARAO PEREIRA DE ARAUJO JUNIOR,"25379,42","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","3677,00","0,00","-5966,33","0,00","-3677,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","19413,09","0,00","1415,12","0,00","0,00","0,00","0,00","0,00","1415,12","0,00"
2025,10,3205874,***.859.807-**,AARAO SOARES,"7005,97","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","-769,05","0,00","-715,34","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","5521,58","0,00","9907,45","0,00","0,00","0,00","0,00","0,00","9907,45","0,00"
2025,10,3449428,***.086.942-**,AARAO TEIXEIRA DOS SANTOS,"6753,20","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","-740,76","0,00","-755,03","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","5257,41","0,00","1542,24","0,00","0,00","0,00","0,00","0,00","1542,24","0,00"


In [0]:
df_servidores_remuneracao = spark.table("servidores.silver.servidores_remuneracao")
display(df_servidores_remuneracao.columns)

_1
ANO
MES
Id_SERVIDOR_PORTAL
CPF
NOME
REMUNERACAO_BASICA_BRUTA_BRL
REMUNERACAO_BASICA_BRUTA_USD
ABATE-TETO_BRL
ABATE-TETO_USD
GRATIFICACAO_NATALINA_BRL


### • Seleção de colunas para análise e modelagem
Seleciona apenas as colunas essenciais para garantir unicidade, rastreabilidade e representatividade dos dados de remuneração dos servidores.

In [0]:
%python
def verificar_colunas_remuneracao_nao_vazias(df):
    """
    Retorna uma lista das colunas de remuneração (terminadas em _USD ou _BRL) que possuem valores válidos.
    Considera como válidos os valores diferentes de '0,00' e None.
    """
    colunas_remuneracao = [
        col for col in df.columns
        if (col.endswith('_USD') or col.endswith('_BRL'))
        and col not in ['ANO', 'MES', 'ID_SERVIDOR_PORTAL', 'NOME']
    ]
    resultado = []
    for col_name in colunas_remuneracao:
        valores_unicos = (
            df.select(col_name)
            .distinct()
            .rdd.flatMap(lambda x: x)
            .collect()
        )
        valores_filtrados = [valor for valor in valores_unicos if valor not in ['0,00', None]]
        if valores_filtrados:
            resultado.append(col_name)
    return resultado

def obter_colunas_remuneracao_vazias(df):
    """
    Retorna uma lista das colunas de remuneração (terminadas em _USD ou _BRL) que estão totalmente vazias,
    ou seja, preenchidas apenas com valores nulos ou '0,00'.
    """
    colunas_remuneracao = [
        col for col in df.columns
        if (col.endswith('_USD') or col.endswith('_BRL'))
        and col not in ['ANO', 'MES', 'ID_SERVIDOR_PORTAL', 'NOME']
    ]
    colunas_vazias = []
    for col in colunas_remuneracao:
        # Conta linhas não nulas e diferentes de '0,00'
        count_not_null = df.filter(
            (df[col].isNotNull()) & (df[col] != '0,00')
        ).count()
        if count_not_null == 0:
            colunas_vazias.append(col)
    return colunas_vazias

In [0]:
# Remoção das colunas 'CPF' e de indenizações militares:
# 'CPF' é excluído por ser dado sensível e não relevante para análise agregada.
# Colunas de indenização militar são removidas por por não se aplicarem ao escopo da análise.

lista_inicial_col_remover = [ ] 
lista_inicial_col_remover = ['CPF','INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_MILITAR_BRL', 'INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_MILITAR_USD']

In [0]:
# Identifica colunas de remuneração totalmente vazias usando funções Spark
lista_remuneracao_col_remover = []

lista_remuneracao_col_remover = obter_colunas_remuneracao_vazias(df_servidores_remuneracao)

# Junta colunas a remover: sensíveis, militares e financeiras vazias
lista_final_remover_cols = []
lista_final_remover_cols = lista_inicial_col_remover + lista_remuneracao_col_remover
print(lista_final_remover_cols)

['CPF', 'INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_MILITAR_BRL', 'INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_MILITAR_USD', 'REMUNERACAO_BASICA_BRUTA_USD', 'ABATE-TETO_USD', 'GRATIFICACAO_NATALINA_USD', 'ABATE-TETO_DA_GRATIFICACAO_NATALINA_BRL', 'ABATE-TETO_DA_GRATIFICACAO_NATALINA_USD', 'FERIAS_USD', 'OUTRAS_REMUNERACOES_EVENTUAIS_USD', 'IRRF_USD', 'PSS/RPGS_USD', 'DEMAIS_DEDUCOES_USD', 'PENSAO_MILITAR_BRL', 'PENSAO_MILITAR_USD', 'FUNDO_DE_SAUDE_BRL', 'FUNDO_DE_SAUDE_USD', 'TAXA_DE_OCUPACAO_IMOVEL_FUNCIONAL_USD', 'REMUNERACAO_APOS_DEDUCOES_OBRIGATORIAS_USD', 'INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_CIVIL_USD', 'INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_MILITAR_BRL', 'INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_MILITAR_USD', 'INDENIZACAO_PROGRAMA_DESLIGAMENTO_VOLUNTARIO_MP_792/2017_USD', 'TOTAL_DE_INDENIZACAO_USD']


In [0]:
df_filtrado_servidor_remuneracao = df_servidores_remuneracao.drop(*lista_final_remover_cols)

df_filtrado_servidor_remuneracao.printSchema()

root
 |-- ANO: string (nullable = true)
 |-- MES: string (nullable = true)
 |-- Id_SERVIDOR_PORTAL: string (nullable = true)
 |-- NOME: string (nullable = true)
 |-- REMUNERACAO_BASICA_BRUTA_BRL: string (nullable = true)
 |-- ABATE-TETO_BRL: string (nullable = true)
 |-- GRATIFICACAO_NATALINA_BRL: string (nullable = true)
 |-- FERIAS_BRL: string (nullable = true)
 |-- OUTRAS_REMUNERACOES_EVENTUAIS_BRL: string (nullable = true)
 |-- IRRF_BRL: string (nullable = true)
 |-- PSS/RPGS_BRL: string (nullable = true)
 |-- DEMAIS_DEDUCOES_BRL: string (nullable = true)
 |-- TAXA_DE_OCUPACAO_IMOVEL_FUNCIONAL_BRL: string (nullable = true)
 |-- REMUNERACAO_APOS_DEDUCOES_OBRIGATORIAS_BRL: string (nullable = true)
 |-- INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_CIVIL_BRL: string (nullable = true)
 |-- INDENIZACAO_PROGRAMA_DESLIGAMENTO_VOLUNTARIO_MP_792/2017_BRL: string (nullable = true)
 |-- TOTAL_DE_INDENIZACAO_BRL: string (nullable = true)



### • Padronização e renomeação dos campos
Renomeia as colunas para o padrão sintético definido na documentação.


In [0]:
mapeamento_colunas_remuneracao = {
    "ANO": "ano",
    "MES": "mes",
    "Id_SERVIDOR_PORTAL": "id",
    "NOME": "nome",
    "REMUNERACAO_BASICA_BRUTA_BRL": "remuneracao_bruta_brl",
    "ABATE-TETO_BRL": "abate_teto_brl",
    "GRATIFICACAO_NATALINA_BRL": "gratificacao_natalina_brl",
    "FERIAS_BRL": "ferias_brl",
    "OUTRAS_REMUNERACOES_EVENTUAIS_BRL": "remuneracoes_eventuais_brl",
    "IRRF_BRL": "irrf_brl",
    "PSS/RPGS_BRL": "pss_rpgs_brl",
    "DEMAIS_DEDUCOES_BRL": "demais_deducoes_brl",
    "TAXA_DE_OCUPACAO_IMOVEL_FUNCIONAL_BRL": "taxa_ocupacao_imovel_funcional_brl",
    "REMUNERACAO_APOS_DEDUCOES_OBRIGATORIAS_BRL": "remuneracao_apos_deducoes_obrigatorias_brl",
    "INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_CIVIL_BRL": "indenizacao_registradas_sistemas_pessoal_civil_brl",
    "INDENIZACAO_PROGRAMA_DESLIGAMENTO_VOLUNTARIO_MP_792/2017_BRL": "indenizacao_programa_desligamento_voluntario_mp_792_2017_brl",
    "TOTAL_DE_INDENIZACAO_BRL": "total_indenizacao_brl"
}

for original, novo in mapeamento_colunas_remuneracao.items():
    if original in df_filtrado_servidor_remuneracao.columns:
        df_filtrado_servidor_remuneracao = df_filtrado_servidor_remuneracao.withColumnRenamed(original, novo)

display(df_filtrado_servidor_remuneracao.limit(5))

ano,mes,id,nome,remuneracao_bruta_brl,abate_teto_brl,gratificacao_natalina_brl,ferias_brl,remuneracoes_eventuais_brl,irrf_brl,pss_rpgs_brl,demais_deducoes_brl,taxa_ocupacao_imovel_funcional_brl,remuneracao_apos_deducoes_obrigatorias_brl,indenizacao_registradas_sistemas_pessoal_civil_brl,indenizacao_programa_desligamento_voluntario_mp_792_2017_brl,total_indenizacao_brl
2025,10,3174964,AARAO CARLOS LUZ MACAMBIRA,"12577,90","0,00","0,00","0,00","0,00","-2112,23","-1592,59","0,00","0,00","8873,08","1294,96","0,00","1294,96"
2025,10,2903139,AARAO FERREIRA LIMA NETO,"28342,89","0,00","0,00","0,00","803,52","-6267,89","-3049,58","0,00","0,00","19828,94","1000,00","0,00","1000,00"
2025,10,3137242,AARAO PEREIRA DE ARAUJO JUNIOR,"25379,42","0,00","0,00","0,00","3677,00","-5966,33","-3677,00","0,00","0,00","19413,09","1415,12","0,00","1415,12"
2025,10,3205874,AARAO SOARES,"7005,97","0,00","0,00","0,00","0,00","-769,05","-715,34","0,00","0,00","5521,58","9907,45","0,00","9907,45"
2025,10,3449428,AARAO TEIXEIRA DOS SANTOS,"6753,20","0,00","0,00","0,00","0,00","-740,76","-755,03","0,00","0,00","5257,41","1542,24","0,00","1542,24"


### • Definição dos campos obrigatórios e opcionais
Define ano, mes, id, valor_remuneracao como obrigatórios; demais campos como opcionais.


### • Conversão e padronização de tipos
Converte valores financeiros para Double.

In [0]:
from pyspark.sql.functions import regexp_replace, col

def converter_colunas_financeiras_para_double(df):
    # Converte apenas colunas que realmente existem e já estão no padrão sintético
    colunas_financeiras = [c for c in df.columns if c.endswith('_brl') or c.endswith('_usd')]
    df_convertido = df
    for coluna in colunas_financeiras:
        if coluna in df_convertido.columns:
            df_convertido = df_convertido.withColumn(
                coluna,
                regexp_replace(col(coluna), ',', '.').cast('double')
            )
    return df_convertido

# Exemplo de uso após renomear as colunas:
df_final_servidor_remuneracao = converter_colunas_financeiras_para_double(df_filtrado_servidor_remuneracao)
df_final_servidor_remuneracao.printSchema()

root
 |-- ano: string (nullable = true)
 |-- mes: string (nullable = true)
 |-- id: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- remuneracao_bruta_brl: double (nullable = true)
 |-- abate_teto_brl: double (nullable = true)
 |-- gratificacao_natalina_brl: double (nullable = true)
 |-- ferias_brl: double (nullable = true)
 |-- remuneracoes_eventuais_brl: double (nullable = true)
 |-- irrf_brl: double (nullable = true)
 |-- pss_rpgs_brl: double (nullable = true)
 |-- demais_deducoes_brl: double (nullable = true)
 |-- taxa_ocupacao_imovel_funcional_brl: double (nullable = true)
 |-- remuneracao_apos_deducoes_obrigatorias_brl: double (nullable = true)
 |-- indenizacao_registradas_sistemas_pessoal_civil_brl: double (nullable = true)
 |-- indenizacao_programa_desligamento_voluntario_mp_792_2017_brl: double (nullable = true)
 |-- total_indenizacao_brl: double (nullable = true)



### • Validação dos dados
Validação de unicidade, consistência entre códigos e descrições, integridade referencial.

In [0]:
from pyspark.sql.functions import col, countDistinct, when, lit

# Validação de unicidade de id com nome
duplicados_id_nome = (
    df_final_servidor_remuneracao
    .groupBy("id")
    .agg(countDistinct("nome").alias("nomes_distintos"))
    .filter(col("nomes_distintos") > 1)
)
display(duplicados_id_nome)

# Validação de consistência entre códigos e descrições (exemplo para id e nome)
# Adicione outros pares conforme necessário
colunas_cod_desc = [
    ("id", "nome"),
]

for cod_col, desc_col in colunas_cod_desc:
    if cod_col in df_final_servidor_remuneracao.columns and desc_col in df_final_servidor_remuneracao.columns:
        inconsistentes = (
            df_final_servidor_remuneracao
            .groupBy(cod_col)
            .agg(countDistinct(desc_col).alias("qtd_desc"))
            .filter(col("qtd_desc") > 1)
        )
        display(inconsistentes)

# Validação de integridade referencial (exemplo: id deve existir em cadastro)
ids_cadastro = spark.table("servidores.silver.servidores_cadastro").select("id").distinct()
ids_remuneracao = df_final_servidor_remuneracao.select("id").distinct()
ids_nao_encontrados = ids_remuneracao.join(ids_cadastro, on="id", how="left_anti")
display(ids_nao_encontrados)

# Validação de campos financeiros: não podem ser nulos, se ausente deve ser 0,00
colunas_financeiras = [c for c in df_final_servidor_remuneracao.columns if c.endswith('_BRL') or c.endswith('_USD')]
for coluna in colunas_financeiras:
    df_final_servidor_remuneracao = df_final_servidor_remuneracao.withColumn(
        coluna,
        when(col(coluna).isNull(), lit(0,00)).otherwise(col(coluna))
    )

# Exibe linhas com valores nulos (após tratamento, não deve haver)
for coluna in colunas_financeiras:
    nulos = df_final_servidor_remuneracao.filter(col(coluna).isNull())
    display(nulos)

id,nomes_distintos


id,qtd_desc


id
null


### • Sobreescrita da tabela tratada
A tabela tratada é sobreescrita após limpeza, validação e padronização.

In [0]:
%python

# Cria tabela Delta tratada e padronizada para remuneração dos servidores
df_final_servidor_remuneracao.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("servidores.silver.servidores_remuneracao")



### • Documentação e comentários no catálogo de dados

In [0]:
%sql

-- Comentário para a tabela servidores_remuneracao
COMMENT ON TABLE servidores.silver.servidores_remuneracao IS 'Tabela de dados de remuneração dos servidores, com informações limpas, padronizadas e prontas para análises de valores, tipos de remuneração, deduções e indenizações.';

-- Comentários para as colunas da tabela servidores_remuneracao
COMMENT ON COLUMN servidores.silver.servidores_remuneracao.ano IS 'Ano de referência da remuneração';
COMMENT ON COLUMN servidores.silver.servidores_remuneracao.mes IS 'Mês de referência da remuneração';
COMMENT ON COLUMN servidores.silver.servidores_remuneracao.id IS 'Identificador único do servidor';
COMMENT ON COLUMN servidores.silver.servidores_remuneracao.nome IS 'Nome do servidor';
COMMENT ON COLUMN servidores.silver.servidores_remuneracao.remuneracao_bruta_brl IS 'Remuneração bruta em reais';
COMMENT ON COLUMN servidores.silver.servidores_remuneracao.abate_teto_brl IS 'Valor abatido do teto constitucional em reais';
COMMENT ON COLUMN servidores.silver.servidores_remuneracao.gratificacao_natalina_brl IS 'Gratificação natalina em reais';
COMMENT ON COLUMN servidores.silver.servidores_remuneracao.ferias_brl IS 'Valor referente a férias em reais';
COMMENT ON COLUMN servidores.silver.servidores_remuneracao.remuneracoes_eventuais_brl IS 'Outras remunerações eventuais em reais';
COMMENT ON COLUMN servidores.silver.servidores_remuneracao.irrf_brl IS 'Imposto de Renda Retido na Fonte em reais';
COMMENT ON COLUMN servidores.silver.servidores_remuneracao.pss_rpgs_brl IS 'Contribuição previdenciária (PSS/RPGS) em reais';
COMMENT ON COLUMN servidores.silver.servidores_remuneracao.demais_deducoes_brl IS 'Demais deduções obrigatórias em reais';
COMMENT ON COLUMN servidores.silver.servidores_remuneracao.taxa_ocupacao_imovel_funcional_brl IS 'Taxa de ocupação de imóvel funcional em reais';
COMMENT ON COLUMN servidores.silver.servidores_remuneracao.remuneracao_apos_deducoes_obrigatorias_brl IS 'Remuneração após deduções obrigatórias em reais';
COMMENT ON COLUMN servidores.silver.servidores_remuneracao.indenizacao_registradas_sistemas_pessoal_civil_brl IS 'Indenizações registradas em sistemas de pessoal civil em reais';
COMMENT ON COLUMN servidores.silver.servidores_remuneracao.indenizacao_programa_desligamento_voluntario_mp_792_2017_brl IS 'Indenização do Programa de Desligamento Voluntário MP 792/2017 em reais';
COMMENT ON COLUMN servidores.silver.servidores_remuneracao.total_indenizacao_brl IS 'Total de indenizações em reais';

DESCRIBE servidores.silver.servidores_remuneracao;

col_name,data_type,comment
ano,string,Ano de referência da remuneração
mes,string,Mês de referência da remuneração
id,string,Identificador único do servidor
nome,string,Nome do servidor
remuneracao_bruta_brl,double,Remuneração bruta em reais
abate_teto_brl,double,Valor abatido do teto constitucional em reais
gratificacao_natalina_brl,double,Gratificação natalina em reais
ferias_brl,double,Valor referente a férias em reais
remuneracoes_eventuais_brl,double,Outras remunerações eventuais em reais
irrf_brl,double,Imposto de Renda Retido na Fonte em reais


### `aposentados_remuneracao`

- **Seleção de colunas para análise e modelagem:**
    - `ano`, `mes`, `id`, `nome`, `remuneracao_bruta_brl`, `abate_teto_brl`, `gratificacao_natalina_brl`, `ferias_brl`, `remuneracoes_eventuais_brl`, `irrf_brl`, `pss_rpgs_brl`, `demais_deducoes_brl`, `taxa_ocupacao_imovel_funcional_brl`, `remuneracao_apos_deducoes_obrigatorias_brl`, `indenizacao_registradas_sistemas_pessoal_civil_brl`, `indenizacao_programa_desligamento_voluntario_mp_792_2017_brl`, `total_indenizacao_brl`.
    - Essas colunas representam as principais medidas e dimensões para análise financeira e administrativa dos aposentados.

- **Padronização e renomeação dos campos:**
    - Colunas originais foram renomeadas para o padrão sintético, facilitando consultas, joins e padronização entre tabelas.
    - Exemplo: `REMUNERACAO_BASICA_BRUTA_BRL` → `remuneracao_bruta_brl`, `FERIAS_BRL` → `ferias_brl`, etc.

- **Definição dos campos obrigatórios e opcionais:**
    - Os campos `ano`, `mes`, `id`, `remuneracao_bruta_brl` são obrigatórios para garantir a identificação única e a integridade dos registros.
    - Os demais campos são opcionais e podem conter valores nulos, preservando a representatividade dos dados originais.

- **Conversão e padronização de tipos:**
    - Valores financeiros convertidos para tipo Double, facilitando cálculos e agregações.
    - Códigos e identificadores convertidos para string.
    - Datas mantidas como string (`ano`, `mes`).

- **Tratamento de valores nulos e ausentes:**
    - Campos financeiros e opcionais podem permanecer nulos, indicando ausência de registro ou informação não aplicável.
    - Valores nulos em campos obrigatórios são tratados e validados.

- **Validação dos dados:**
    - Garantia de unicidade em `id`, `ano`, `mes`.
    - Checagem de consistência entre identificadores e nomes.
    - Validação de integridade referencial entre `id` e cadastro de aposentados.
    - Verificação de ausência de duplicidades e registros inconsistentes.
    - Conferência de domínios esperados para atributos categóricos e financeiros.

- **Sobreescrita da tabela tratada:**
    - A tabela `aposentados_remuneracao` tratada é sobreescrita após o processo de limpeza, validação e padronização, garantindo que apenas registros válidos e consistentes sejam mantidos para análise.

- **Documentação e comentários no catálogo de dados:**
    - Comentários detalhados foram adicionados para as colunas e para a tabela no catálogo de dados, explicando o significado, uso e regras de negócio de cada atributo.

### • Seleção de colunas para análise e modelagem
Seleciona apenas as colunas essenciais para garantir unicidade, rastreabilidade e representatividade dos dados de remuneração dos aposentados.

In [0]:
%sql
select * from servidores.silver.aposentados_remuneracao limit 5

ANO,MES,Id_SERVIDOR_PORTAL,CPF,NOME,REMUNERACAO_BASICA_BRUTA_BRL,REMUNERACAO_BASICA_BRUTA_USD,ABATE-TETO_BRL,ABATE-TETO_USD,GRATIFICACAO_NATALINA_BRL,GRATIFICACAO_NATALINA_USD,ABATE-TETO_DA_GRATIFICACAO_NATALINA_BRL,ABATE-TETO_DA_GRATIFICACAO_NATALINA_USD,FERIAS_BRL,FERIAS_USD,OUTRAS_REMUNERACOES_EVENTUAIS_BRL,OUTRAS_REMUNERACOES_EVENTUAIS_USD,IRRF_BRL,IRRF_USD,PSS/RPGS_BRL,PSS/RPGS_USD,DEMAIS_DEDUCOES_BRL,DEMAIS_DEDUCOES_USD,PENSAO_MILITAR_BRL,PENSAO_MILITAR_USD,FUNDO_DE_SAUDE_BRL,FUNDO_DE_SAUDE_USD,TAXA_DE_OCUPACAO_IMOVEL_FUNCIONAL_BRL,TAXA_DE_OCUPACAO_IMOVEL_FUNCIONAL_USD,REMUNERACAO_APOS_DEDUCOES_OBRIGATORIAS_BRL,REMUNERACAO_APOS_DEDUCOES_OBRIGATORIAS_USD,INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_CIVIL_BRL,INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_CIVIL_USD,INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_MILITAR_BRL,INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_MILITAR_USD,INDENIZACAO_PROGRAMA_DESLIGAMENTO_VOLUNTARIO_MP_792/2017_BRL,INDENIZACAO_PROGRAMA_DESLIGAMENTO_VOLUNTARIO_MP_792/2017_USD,TOTAL_DE_INDENIZACAO_BRL,TOTAL_DE_INDENIZACAO_USD
2025,10,2025221,***.559.144-**,AARAO DE ANDRADE LIMA,"25227,84","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","323,26","0,00","-4710,59","0,00","-2700,37","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","18140,14","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00"
2025,10,3594060,***.924.486-**,AARAO MOREIRA DA SILVA,"5782,26","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","-96,50","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","5685,76","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00"
2025,10,87244,***.681.016-**,ABA ISRAEL COHEN PERSIANO,"26113,50","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","-2846,51","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","23266,99","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00"
2025,10,661149,***.430.802-**,ABADIA APARECIDA FAUSTINO,"2519,19","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","2519,19","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00"
2025,10,3336973,***.078.406-**,ABADIA BELCHIOR GOMES,"4075,23","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","321,04","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","4396,27","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00"


In [0]:
df_aposentados_remuneracao = spark.table("servidores.silver.aposentados_remuneracao")
display(df_aposentados_remuneracao.columns)

_1
ANO
MES
Id_SERVIDOR_PORTAL
CPF
NOME
REMUNERACAO_BASICA_BRUTA_BRL
REMUNERACAO_BASICA_BRUTA_USD
ABATE-TETO_BRL
ABATE-TETO_USD
GRATIFICACAO_NATALINA_BRL


### • Seleção de colunas para análise e modelagem
Seleciona apenas as colunas essenciais para garantir unicidade, rastreabilidade e representatividade dos dados de remuneração dos aposentados.

In [0]:
colunas_selecionadas = [col for col in [
    "ANO", "MES", "Id_SERVIDOR_PORTAL", "NOME", "REMUNERACAO_BASICA_BRUTA_BRL", "ABATE-TETO_BRL", "GRATIFICACAO_NATALINA_BRL", "FERIAS_BRL", "OUTRAS_REMUNERACOES_EVENTUAIS_BRL", "IRRF_BRL", "PSS/RPGS_BRL", "DEMAIS_DEDUCOES_BRL", "TAXA_DE_OCUPACAO_IMOVEL_FUNCIONAL_BRL", "REMUNERACAO_APOS_DEDUCOES_OBRIGATORIAS_BRL", "INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_CIVIL_BRL", "INDENIZACAO_PROGRAMA_DESLIGAMENTO_VOLUNTARIO_MP_792/2017_BRL", "TOTAL_DE_INDENIZACAO_BRL"
] if col in df_aposentados_remuneracao.columns]
df_aposentados_remuneracao = df_aposentados_remuneracao.select(*colunas_selecionadas)
display(df_aposentados_remuneracao.limit(5))

ANO,MES,Id_SERVIDOR_PORTAL,NOME,REMUNERACAO_BASICA_BRUTA_BRL,ABATE-TETO_BRL,GRATIFICACAO_NATALINA_BRL,FERIAS_BRL,OUTRAS_REMUNERACOES_EVENTUAIS_BRL,IRRF_BRL,PSS/RPGS_BRL,DEMAIS_DEDUCOES_BRL,TAXA_DE_OCUPACAO_IMOVEL_FUNCIONAL_BRL,REMUNERACAO_APOS_DEDUCOES_OBRIGATORIAS_BRL,INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_CIVIL_BRL,INDENIZACAO_PROGRAMA_DESLIGAMENTO_VOLUNTARIO_MP_792/2017_BRL,TOTAL_DE_INDENIZACAO_BRL
2025,10,2025221,AARAO DE ANDRADE LIMA,"25227,84","0,00","0,00","0,00","323,26","-4710,59","-2700,37","0,00","0,00","18140,14","0,00","0,00","0,00"
2025,10,3594060,AARAO MOREIRA DA SILVA,"5782,26","0,00","0,00","0,00","0,00","-96,50","0,00","0,00","0,00","5685,76","0,00","0,00","0,00"
2025,10,87244,ABA ISRAEL COHEN PERSIANO,"26113,50","0,00","0,00","0,00","0,00","0,00","-2846,51","0,00","0,00","23266,99","0,00","0,00","0,00"
2025,10,661149,ABADIA APARECIDA FAUSTINO,"2519,19","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","2519,19","0,00","0,00","0,00"
2025,10,3336973,ABADIA BELCHIOR GOMES,"4075,23","0,00","0,00","0,00","321,04","0,00","0,00","0,00","0,00","4396,27","0,00","0,00","0,00"


### • Remoção de colunas sensíveis e irrelevantes
Remove colunas sensíveis (CPF) e militares, além de colunas de remuneração totalmente vazias.

In [0]:
lista_colunas_remover = ['CPF','INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_MILITAR_BRL', 'INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_MILITAR_USD']
lista_remuneracao_col_remover = obter_colunas_remuneracao_vazias(df_aposentados_remuneracao)
lista_final_remover_cols = lista_colunas_remover + lista_remuneracao_col_remover
df_filtrado_aposentados_remuneracao = df_aposentados_remuneracao.drop(*lista_final_remover_cols)
display(df_filtrado_aposentados_remuneracao.limit(5))

ANO,MES,Id_SERVIDOR_PORTAL,NOME,REMUNERACAO_BASICA_BRUTA_BRL,ABATE-TETO_BRL,GRATIFICACAO_NATALINA_BRL,FERIAS_BRL,OUTRAS_REMUNERACOES_EVENTUAIS_BRL,IRRF_BRL,PSS/RPGS_BRL,DEMAIS_DEDUCOES_BRL,REMUNERACAO_APOS_DEDUCOES_OBRIGATORIAS_BRL,INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_CIVIL_BRL,TOTAL_DE_INDENIZACAO_BRL
2025,10,2025221,AARAO DE ANDRADE LIMA,"25227,84","0,00","0,00","0,00","323,26","-4710,59","-2700,37","0,00","18140,14","0,00","0,00"
2025,10,3594060,AARAO MOREIRA DA SILVA,"5782,26","0,00","0,00","0,00","0,00","-96,50","0,00","0,00","5685,76","0,00","0,00"
2025,10,87244,ABA ISRAEL COHEN PERSIANO,"26113,50","0,00","0,00","0,00","0,00","0,00","-2846,51","0,00","23266,99","0,00","0,00"
2025,10,661149,ABADIA APARECIDA FAUSTINO,"2519,19","0,00","0,00","0,00","0,00","0,00","0,00","0,00","2519,19","0,00","0,00"
2025,10,3336973,ABADIA BELCHIOR GOMES,"4075,23","0,00","0,00","0,00","321,04","0,00","0,00","0,00","4396,27","0,00","0,00"


### • Padronização e renomeação dos campos
Renomeia as colunas para o padrão sintético definido na documentação.

In [0]:
mapeamento_colunas_remuneracao = {
    "ANO": "ano",
    "MES": "mes",
    "Id_SERVIDOR_PORTAL": "id",
    "NOME": "nome",
    "REMUNERACAO_BASICA_BRUTA_BRL": "remuneracao_bruta_brl",
    "ABATE-TETO_BRL": "abate_teto_brl",
    "GRATIFICACAO_NATALINA_BRL": "gratificacao_natalina_brl",
    "FERIAS_BRL": "ferias_brl",
    "OUTRAS_REMUNERACOES_EVENTUAIS_BRL": "remuneracoes_eventuais_brl",
    "IRRF_BRL": "irrf_brl",
    "PSS/RPGS_BRL": "pss_rpgs_brl",
    "DEMAIS_DEDUCOES_BRL": "demais_deducoes_brl",
    "TAXA_DE_OCUPACAO_IMOVEL_FUNCIONAL_BRL": "taxa_ocupacao_imovel_funcional_brl",
    "REMUNERACAO_APOS_DEDUCOES_OBRIGATORIAS_BRL": "remuneracao_apos_deducoes_obrigatorias_brl",
    "INDENIZACAO_REGISTRADAS_EM_SISTEMAS_DE_PESSOAL_-_CIVIL_BRL": "indenizacao_registradas_sistemas_pessoal_civil_brl",
    "INDENIZACAO_PROGRAMA_DESLIGAMENTO_VOLUNTARIO_MP_792/2017_BRL": "indenizacao_programa_desligamento_voluntario_mp_792_2017_brl",
    "TOTAL_DE_INDENIZACAO_BRL": "total_indenizacao_brl"
}
for original, novo in mapeamento_colunas_remuneracao.items():
    if original in df_filtrado_aposentados_remuneracao.columns:
        df_filtrado_aposentados_remuneracao = df_filtrado_aposentados_remuneracao.withColumnRenamed(original, novo)
display(df_filtrado_aposentados_remuneracao.limit(5))

ano,mes,id,nome,remuneracao_bruta_brl,abate_teto_brl,gratificacao_natalina_brl,ferias_brl,remuneracoes_eventuais_brl,irrf_brl,pss_rpgs_brl,demais_deducoes_brl,remuneracao_apos_deducoes_obrigatorias_brl,indenizacao_registradas_sistemas_pessoal_civil_brl,total_indenizacao_brl
2025,10,2025221,AARAO DE ANDRADE LIMA,"25227,84","0,00","0,00","0,00","323,26","-4710,59","-2700,37","0,00","18140,14","0,00","0,00"
2025,10,3594060,AARAO MOREIRA DA SILVA,"5782,26","0,00","0,00","0,00","0,00","-96,50","0,00","0,00","5685,76","0,00","0,00"
2025,10,87244,ABA ISRAEL COHEN PERSIANO,"26113,50","0,00","0,00","0,00","0,00","0,00","-2846,51","0,00","23266,99","0,00","0,00"
2025,10,661149,ABADIA APARECIDA FAUSTINO,"2519,19","0,00","0,00","0,00","0,00","0,00","0,00","0,00","2519,19","0,00","0,00"
2025,10,3336973,ABADIA BELCHIOR GOMES,"4075,23","0,00","0,00","0,00","321,04","0,00","0,00","0,00","4396,27","0,00","0,00"


### • Definição dos campos obrigatórios e opcionais
Define ano, mes, id, remuneracao_bruta_brl como obrigatórios; demais campos como opcionais.

### • Conversão e padronização de tipos

In [0]:

# Converte colunas financeiras para tipo double 
df_final_aposentados_remuneracao = converter_colunas_financeiras_para_double(df_filtrado_aposentados_remuneracao)

df_final_aposentados_remuneracao.printSchema()

root
 |-- ano: string (nullable = true)
 |-- mes: string (nullable = true)
 |-- id: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- remuneracao_bruta_brl: double (nullable = true)
 |-- abate_teto_brl: double (nullable = true)
 |-- gratificacao_natalina_brl: double (nullable = true)
 |-- ferias_brl: double (nullable = true)
 |-- remuneracoes_eventuais_brl: double (nullable = true)
 |-- irrf_brl: double (nullable = true)
 |-- pss_rpgs_brl: double (nullable = true)
 |-- demais_deducoes_brl: double (nullable = true)
 |-- remuneracao_apos_deducoes_obrigatorias_brl: double (nullable = true)
 |-- indenizacao_registradas_sistemas_pessoal_civil_brl: double (nullable = true)
 |-- total_indenizacao_brl: double (nullable = true)



### • Validação dos dados
Validação de unicidade, consistência entre identificadores e nomes, integridade referencial.

In [0]:
from pyspark.sql.functions import col, countDistinct, when, lit

# Validação de unicidade de id com nome
duplicados_id_nome = (
    df_final_aposentados_remuneracao
    .groupBy("id")
    .agg(countDistinct("nome").alias("nomes_distintos"))
    .filter(col("nomes_distintos") > 1)
)
display(duplicados_id_nome)

# Validação de integridade referencial (id deve existir em cadastro)
ids_cadastro = spark.table("servidores.silver.aposentados_cadastro").select("id").distinct()
ids_remuneracao = df_final_aposentados_remuneracao.select("id").distinct()
ids_nao_encontrados = ids_remuneracao.join(ids_cadastro, on="id", how="left_anti")
display(ids_nao_encontrados)

# Validação de campos financeiros: não podem ser nulos, se ausente deve ser 0.00
colunas_financeiras = [c for c in df_final_aposentados_remuneracao.columns if c.endswith('_BRL') or c.endswith('_USD')]
for coluna in colunas_financeiras:
    df_final_aposentados_remuneracao = df_final_aposentados_remuneracao.withColumn(
        coluna,
        when(col(coluna).isNull(), lit(0.00)).otherwise(col(coluna))
    )

# Exibe linhas com valores nulos (após tratamento, não deve haver)
for coluna in colunas_financeiras:
    nulos = df_final_aposentados_remuneracao.filter(col(coluna).isNull())
    display(nulos)

id,nomes_distintos


id
null


### • Sobreescrita da tabela tratada
A tabela tratada é sobreescrita após limpeza, validação e padronização.

In [0]:
df_final_aposentados_remuneracao.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("servidores.silver.aposentados_remuneracao")

### • Documentação e comentários no catálogo de dados
Inclui comentários detalhados para cada coluna e para a tabela, explicando o significado, uso e regras de negócio de cada atributo.

In [0]:
%sql
COMMENT ON TABLE servidores.silver.aposentados_remuneracao IS 'Tabela de dados de remuneração dos aposentados, com informações limpas, padronizadas e prontas para análises de valores, tipos de remuneração, deduções e indenizações.';

COMMENT ON COLUMN servidores.silver.aposentados_remuneracao.ano IS 'Ano de referência da remuneração';
COMMENT ON COLUMN servidores.silver.aposentados_remuneracao.mes IS 'Mês de referência da remuneração';
COMMENT ON COLUMN servidores.silver.aposentados_remuneracao.id IS 'Identificador único do servidor aposentado';
COMMENT ON COLUMN servidores.silver.aposentados_remuneracao.nome IS 'Nome do aposentado';
COMMENT ON COLUMN servidores.silver.aposentados_remuneracao.remuneracao_bruta_brl IS 'Remuneração bruta em reais';
COMMENT ON COLUMN servidores.silver.aposentados_remuneracao.abate_teto_brl IS 'Valor abatido do teto constitucional em reais';
COMMENT ON COLUMN servidores.silver.aposentados_remuneracao.gratificacao_natalina_brl IS 'Gratificação natalina em reais';
COMMENT ON COLUMN servidores.silver.aposentados_remuneracao.ferias_brl IS 'Valor referente a férias em reais';
COMMENT ON COLUMN servidores.silver.aposentados_remuneracao.remuneracoes_eventuais_brl IS 'Outras remunerações eventuais em reais';
COMMENT ON COLUMN servidores.silver.aposentados_remuneracao.irrf_brl IS 'Imposto de Renda Retido na Fonte em reais';
COMMENT ON COLUMN servidores.silver.aposentados_remuneracao.pss_rpgs_brl IS 'Contribuição previdenciária (PSS/RPGS) em reais';
COMMENT ON COLUMN servidores.silver.aposentados_remuneracao.demais_deducoes_brl IS 'Demais deduções obrigatórias em reais';
COMMENT ON COLUMN servidores.silver.aposentados_remuneracao.remuneracao_apos_deducoes_obrigatorias_brl IS 'Remuneração após deduções obrigatórias em reais';
COMMENT ON COLUMN servidores.silver.aposentados_remuneracao.indenizacao_registradas_sistemas_pessoal_civil_brl IS 'Indenizações registradas em sistemas de pessoal civil em reais';
COMMENT ON COLUMN servidores.silver.aposentados_remuneracao.total_indenizacao_brl IS 'Total de indenizações em reais';

In [0]:
%sql
DESCRIBE servidores.silver.aposentados_remuneracao

col_name,data_type,comment
ano,string,Ano de referência da remuneração
mes,string,Mês de referência da remuneração
id,string,Identificador único do servidor aposentado
nome,string,Nome do aposentado
remuneracao_bruta_brl,double,Remuneração bruta em reais
abate_teto_brl,double,Valor abatido do teto constitucional em reais
gratificacao_natalina_brl,double,Gratificação natalina em reais
ferias_brl,double,Valor referente a férias em reais
remuneracoes_eventuais_brl,double,Outras remunerações eventuais em reais
irrf_brl,double,Imposto de Renda Retido na Fonte em reais


In [0]:
%sql
DESCRIBE DETAIL aposentados_remuneracao;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,a0679133-2835-4497-ae64-57a699d76dc0,servidores.silver.aposentados_remuneracao,"Tabela de dados de remuneração dos aposentados, com informações limpas, padronizadas e prontas para análises de valores, tipos de remuneração, deduções e indenizações.",,2025-12-22T04:17:46.035Z,2025-12-22T04:19:59.000Z,List(),List(),1,10956553,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


# 5.3) Camada Gold

Nesta etapa, os dados tratados e integrados das camadas anteriores são consolidados em modelos analíticos prontos para consumo.  
Além das agregações, cálculos de indicadores, enriquecimento com dimensões e preparação dos dados para visualizações e análises avançadas, será incluído o modelo de dados nesta tabela, visando responder diretamente às perguntas de negócio do MVP.

# Esquema Estrela - Camada GOLD

O modelo dimensional implementado na camada GOLD segue o padrão estrela, facilitando análises e agregações. As principais tabelas são:

* **Fato Servidor**: Centraliza os vínculos, cargos, atividades, órgãos, situação, datas e localidade dos servidores.
* **Dimensões**: Cargo, Atividade, Órgão, Vínculo, Tempo, Localidade, Pessoa.

Cada dimensão pode ser usada para cruzamentos, filtros e agrupamentos, respondendo às perguntas do objetivo do trabalho. O modelo garante integridade, performance e flexibilidade para análises exploratórias e dashboards.

###Esquema Estrela

O modelo dimensional implementado na camada GOLD segue o padrão estrela, facilitando análises e agregações. As principais tabelas são:

- **Fato Servidor**: Centraliza os vínculos, cargos, atividades, órgãos, situação, datas e localidade dos servidores.
- **Fato Remuneração**: Consolida todas as medidas financeiras dos servidores, permitindo análises por período, tipo de remuneração, deduções e indenizações. Ela se conecta às dimensões de tempo, pessoa e demais dimensões do modelo estrela, viabilizando cruzamentos e agregações para responder perguntas sobre valores pagos, variações ao longo do tempo e comparativos entre grupos de servidores.
- **Dimensões**: Cargo, Atividade, Órgão, Vínculo, Tempo, Localidade, Pessoa.

Cada dimensão pode ser usada para cruzamentos, filtros e agrupamentos, respondendo às perguntas do objetivo do trabalho. O modelo garante integridade, performance e flexibilidade para análises exploratórias, dashboards e comparativos financeiros entre grupos de servidores.

In [0]:
%sql
-- Dimensão Cargo
CREATE OR REPLACE TABLE servidores.gold.dim_cargo AS
SELECT DISTINCT cod_cargo AS cod_cargo, desc_cargo AS desc_cargo
FROM servidores.silver.servidores_cadastro
WHERE cod_cargo IS NOT NULL AND desc_cargo IS NOT NULL
UNION
SELECT DISTINCT cod_cargo AS cod_cargo, desc_cargo AS desc_cargo
FROM servidores.silver.aposentados_cadastro
WHERE cod_cargo IS NOT NULL AND desc_cargo IS NOT NULL;

-- Dimensão Atividade
CREATE OR REPLACE TABLE servidores.gold.dim_atividade AS
SELECT DISTINCT cod_atividade AS cod_atividade, desc_atividade AS desc_atividade
FROM servidores.silver.servidores_cadastro
WHERE cod_atividade IS NOT NULL AND desc_atividade IS NOT NULL;

-- Dimensão Órgão
CREATE OR REPLACE TABLE servidores.gold.dim_orgao AS
SELECT DISTINCT cod_org_lotacao AS cod_org_lotacao, org_lotacao AS org_lotacao, cod_orgsup_lotacao AS cod_orgsup_lotacao, orgsup_lotacao AS orgsup_lotacao
FROM servidores.silver.servidores_cadastro
WHERE cod_org_lotacao IS NOT NULL AND org_lotacao IS NOT NULL
UNION
SELECT DISTINCT cod_org_lotacao AS cod_org_lotacao, org_lotacao AS org_lotacao, cod_orgsup_lotacao AS cod_orgsup_lotacao, orgsup_lotacao AS orgsup_lotacao
FROM servidores.silver.aposentados_cadastro
WHERE cod_org_lotacao IS NOT NULL AND org_lotacao IS NOT NULL;

-- Dimensão Vínculo
CREATE OR REPLACE TABLE servidores.gold.dim_vinculo AS
SELECT DISTINCT cod_situacao_vinculo AS cod_situacao_vinculo, situacao_vinculo AS situacao_vinculo
FROM servidores.silver.servidores_cadastro
WHERE cod_situacao_vinculo IS NOT NULL AND situacao_vinculo IS NOT NULL;

-- Dimensão Tipo Aposentadoria
CREATE OR REPLACE TABLE servidores.gold.dim_tipo_aposentadoria AS
SELECT DISTINCT cod_tipo_aposentadoria AS cod_tipo_aposentadoria, tipo_aposentadoria AS tipo_aposentadoria
FROM servidores.silver.aposentados_cadastro
WHERE cod_tipo_aposentadoria IS NOT NULL AND tipo_aposentadoria IS NOT NULL;

-- Dimensão Data Aposentadoria
CREATE OR REPLACE TABLE servidores.gold.dim_data_aposentadoria AS
SELECT DISTINCT 
  CASE 
    WHEN dt_aposentadoria IS NULL OR TRIM(CAST(dt_aposentadoria AS STRING)) = '' THEN NULL
    ELSE dt_aposentadoria
  END AS dt_aposentadoria
FROM servidores.silver.aposentados_cadastro;

-- Dimensão Data Afastamento
CREATE OR REPLACE TABLE servidores.gold.dim_data_afastamento AS
SELECT DISTINCT dt_inicio_afastamento AS dt_inicio_afastamento, dt_fim_afastamento AS dt_fim_afastamento
FROM servidores.silver.servidores_cadastro
WHERE dt_inicio_afastamento IS NOT NULL OR dt_fim_afastamento IS NOT NULL;

-- Dimensão Localidade
CREATE OR REPLACE TABLE servidores.gold.dim_localidade AS
SELECT DISTINCT uf_exercicio AS uf_exercicio
FROM servidores.silver.servidores_cadastro
WHERE uf_exercicio IS NOT NULL;

-- Dimensão Pessoa
CREATE OR REPLACE TABLE servidores.gold.dim_pessoa AS
SELECT DISTINCT id AS id, nome AS nome
FROM servidores.silver.servidores_cadastro
WHERE id IS NOT NULL AND nome IS NOT NULL
UNION
SELECT DISTINCT id AS id, nome AS nome
FROM servidores.silver.aposentados_cadastro
WHERE id IS NOT NULL AND nome IS NOT NULL;

num_affected_rows,num_inserted_rows


In [0]:
for table in spark.catalog.listTables("servidores.gold"):
    if table.tableType == "MANAGED" and table.name.startswith("dim_"):
        df = spark.table(f"servidores.gold.{table.name}")
        display(df.limit(5))

id,nome
2209017,ABADIO PEREIRA DAS VIRGES
2175389,ADAIL OLIVEIRA SANTOS
2046951,ADELAIDE CARNEIRO DOS SANTOS
216966,AILTON DE ARRUDA
3200590,AILTON DIAS DE OLIVEIRA


cod_atividade,desc_atividade
5014,COORDENADOR(A)GERAL ADJUNTO(A)
1395,GERENTE II
0172,CHEFE ASSES/COMUNICACAO SOCIAL
5022,GERENTE NIVEL III
1284,GERENTE AREA REGIONAL GF-VIII


cod_cargo,desc_cargo
1806,PUBLICITARIO
2133,TECNICO EM AS EDUCACIONAIS
520,ASSISTENTE DE SEGURANCA
1419,MONITOR - NI
124,AGENTE DE SERVICOS GERAIS


dt_inicio_afastamento,dt_fim_afastamento


dt_aposentadoria
2009-03-19
2009-06-02
2004-02-03
2022-10-28
2005-05-11


...

id,nome
1256921,ABELARDO BENTO ARAUJO
2674886,ADELIA ROCHA SIMEONI
3117257,ADEMAR ALVES FERREIRA
543200,ADEMAR PASSOS DE OLIVEIRA SEGUNDO
501738,ADILSON DO VALE GOMES


id,nome
1256921,ABELARDO BENTO ARAUJO
2674886,ADELIA ROCHA SIMEONI
3117257,ADEMAR ALVES FERREIRA
543200,ADEMAR PASSOS DE OLIVEIRA SEGUNDO
501738,ADILSON DO VALE GOMES


DATA_INICIO_AFASTAMENTO,DATA_TERMINO_AFASTAMENTO,DATA_APOSENTADORIA
null,null,2009-03-19
null,null,2009-06-02
null,null,2004-02-03
null,null,2022-10-28
null,null,2005-05-11


cod_tipo_aposentadoria,tipo_aposentadoria
04,OUTROS
03,APOSENTADORIA COMPULSORIA
01,APOSENTADORIA VOLUNTARIA
08,APOSENTADORIA POR INCAPACIDADE
02,APOSENTADORIA POR INVALIDEZ


cod_situacao_vinculo,situacao_vinculo
4,ATIVO EM OUTRO ORGAO
15,CLT ANS -DEC 6657/08
39,MIL/DIARIA ASILADO
19,COLABORADOR ICT
20,CONSELHEIRO DO CARF


In [0]:
%sql
-- Tabela Fato Servidor
CREATE OR REPLACE TABLE servidores.gold.fato_servidor AS
SELECT 
  sc.id,
  sc.nome,
  sc.cod_cargo,
  sc.cod_atividade,
  sc.cod_org_lotacao,
  sc.cod_orgsup_lotacao,
  sc.cod_situacao_vinculo,
  sc.situacao_vinculo,
  sc.dt_inicio_afastamento,
  sc.dt_fim_afastamento,
  NULL AS dt_aposentadoria, -- coluna não existe na servidores_cadastro
  sc.uf_exercicio
FROM servidores.silver.servidores_cadastro sc
WHERE sc.id IS NOT NULL AND sc.nome IS NOT NULL;

-- Tabela Fato Remuneração
CREATE OR REPLACE TABLE servidores.gold.fato_remuneracao AS
SELECT
  sr.ano,
  sr.mes,
  sr.id,
  sr.nome,
  sr.remuneracao_bruta_brl,
  sr.abate_teto_brl,
  sr.gratificacao_natalina_brl,
  sr.ferias_brl,
  sr.remuneracoes_eventuais_brl,
  sr.irrf_brl,
  sr.pss_rpgs_brl,
  sr.demais_deducoes_brl,
  sr.taxa_ocupacao_imovel_funcional_brl,
  sr.remuneracao_apos_deducoes_obrigatorias_brl,
  sr.indenizacao_registradas_sistemas_pessoal_civil_brl,
  sr.indenizacao_programa_desligamento_voluntario_mp_792_2017_brl,
  sr.total_indenizacao_brl
FROM servidores.silver.servidores_remuneracao sr
WHERE sr.id IS NOT NULL AND sr.nome IS NOT NULL;

num_affected_rows,num_inserted_rows


In [0]:
# Criação das tabelas dimensão (dim) para o modelo dimensional

# Dimensão de servidores
df_dim_servidor = spark.table("servidores.silver.servidores_cadastro").select("id", "nome").distinct()
df_dim_servidor.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("servidores.gold.dim_servidor")

# Dimensão de aposentados
df_dim_aposentado = spark.table("servidores.silver.aposentados_cadastro").select("id", "nome").distinct()
df_dim_aposentado.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("servidores.gold.dim_aposentado")

# Dimensão de cargos
df_dim_cargo = spark.table("servidores.silver.servidores_cadastro").select("cod_cargo", "desc_cargo").distinct()
df_dim_cargo.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("servidores.gold.dim_cargo")

# Dimensão de órgãos de lotação
df_dim_org_lotacao = spark.table("servidores.silver.servidores_cadastro").select("cod_org_lotacao", "org_lotacao").distinct()
df_dim_org_lotacao.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("servidores.gold.dim_org_lotacao")

# Dimensão de órgãos superiores de lotação
df_dim_orgsup_lotacao = spark.table("servidores.silver.servidores_cadastro").select("cod_orgsup_lotacao", "orgsup_lotacao").distinct()
df_dim_orgsup_lotacao.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("servidores.gold.dim_orgsup_lotacao")

# Dimensão de tipo de aposentadoria
df_dim_tipo_aposentadoria = spark.table("servidores.silver.aposentados_cadastro").select("cod_tipo_aposentadoria", "tipo_aposentadoria").distinct()
df_dim_tipo_aposentadoria.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("servidores.gold.dim_tipo_aposentadoria")

In [0]:
# Criação da tabela fato_servidor
df_fato_servidor = spark.table("servidores.silver.servidores_cadastro")
df_fato_servidor.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("servidores.gold.fato_servidor")

# Criação da tabela fato_remuneracao
df_fato_remuneracao = spark.table("servidores.silver.servidores_remuneracao")
df_fato_remuneracao.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("servidores.gold.fato_remuneracao")

### 7. Análise - Qualidade

Principais problemas de qualidade identificados e resolvidos nos dados do CSV do Portal da Transparência:

- Presença de valores nulos e inválidos em campos essenciais (ex: datas, códigos, descrições).
- Formatos de data inconsistentes (ex: dd/MM/yyyy misturado com yyyy-MM-dd).
- Códigos e descrições sem pareamento correto, exigindo geração de identificadores artificiais.
- Colunas sensíveis e irrelevantes (ex: CPF, campos militares) removidas para garantir privacidade e foco analítico.
- Valores financeiros em formato string com vírgula, convertidos para double.
- Registros duplicados e inconsistentes eliminados via validação de unicidade e integridade referencial.
- Domínios categóricos revisados para garantir padronização e evitar erros em agregações.

Esses ajustes garantiram que a base final na camada GOLD esteja pronta para análises confiáveis e comparáveis.

### 8. Análise - Solução

**Pergunta 1:** Qual o total de servidores ativos por órgão?


In [0]:
%sql
SELECT org_lotacao, COUNT(*) AS total_servidores
FROM servidores.gold.fato_servidor
GROUP BY org_lotacao
ORDER BY total_servidores DESC;

org_lotacao,total_servidores
Ministério da Saúde,62110
Empresa Brasileira de Serviços Hospitalares,53771
MINISTERIO DA FAZENDA,25221
Instituto Nacional do Seguro Social,23622
Departamento de Polícia Federal,16317
Universidade Federal do Rio de Janeiro,15346
MIN GESTAO E INOV EM SERV PUBLICOS,15187
Departamento de Polícia Rodoviária Federal,14162
Hospital Nossa Senhora da Conceição,13858
Fundação Instituto Brasileiro de Geografia e Estatística,13462


**Pergunta 1:** Qual o quantitativo total de servidores (ativos e inativos) no governo federal?

In [0]:
sql_total_servidores = """
SELECT COUNT(DISTINCT id) AS total_servidores
FROM servidores.gold.fato_servidor
"""
display(spark.sql(sql_total_servidores))

total_servidores
630557


**Pergunta 2:** Qual a distribuição percentual dos tipos de vínculo dos servidores na ativa (e.g., efetivos/concursados, comissionados, temporários)?

> **Comentário:** Esta análise foi excluída porque os dados de vínculos estavam sem integridade com os códigos e situação_vinculo.

**Pergunta 3:** Qual o quantitativo de servidores em situação de afastamento ou licença?

> **Comentário:** As colunas de data de início e fim de afastamento estavam totalmente nulas nas origens de dados, impossibilitando análises temporais sobre afastamentos.



In [0]:
%sql
SELECT 
  COUNT(DISTINCT id) AS qtd_afastados_licenca
FROM servidores.gold.fato_servidor
WHERE situacao_vinculo IN ('AFASTADO', 'LICENÇA')

qtd_afastados_licenca
0


**Pergunta 4:** Dos servidores afastados, quantos são por licença-saúde, licença para interesse/capacitação ou licença-prêmio?

**Comentário: Não existe essa classificação para os servidores afastado no conjunto de dados.**

**Pergunta 5:** Qual o Órgão que possui o maior quantitativo de servidores?

In [0]:
sql_orgao_maior_qtd = """
SELECT 
  org_lotacao,
  COUNT(DISTINCT id) AS qtd_servidores
FROM servidores.gold.fato_servidor
GROUP BY org_lotacao
ORDER BY qtd_servidores DESC
LIMIT 1
"""
display(spark.sql(sql_orgao_maior_qtd))

org_lotacao,qtd_servidores
Ministério da Saúde,60319


**Pergunta 6:** Qual a carreira/cargo com o maior quantitativo de servidores?

In [0]:
sql_cargo_maior_qtd = """
SELECT 
  desc_cargo,
  COUNT(DISTINCT id) AS qtd_servidores
FROM servidores.gold.fato_servidor
WHERE desc_cargo IS NOT NULL
GROUP BY desc_cargo
ORDER BY qtd_servidores DESC
LIMIT 1
"""
display(spark.sql(sql_cargo_maior_qtd))

desc_cargo,qtd_servidores
PROFESSOR DO MAGISTERIO SUPERIOR,88154


**Pergunta 7:** Qual o Órgão com a maior remuneração média?

In [0]:
try:
    sql_orgao_maior_media_remuneracao = """
    SELECT s.org_lotacao, ROUND(AVG(r.remuneracao_apos_deducoes_obrigatorias_brl), 2) AS media_remuneracao
    FROM servidores.gold.fato_remuneracao r
    JOIN servidores.gold.fato_servidor s ON r.id = s.id
    GROUP BY s.org_lotacao
    ORDER BY media_remuneracao DESC
    LIMIT 1
    """
    display(spark.sql(sql_orgao_maior_media_remuneracao))
except Exception as e:
    print(f"Erro ao executar a consulta: {e}")

org_lotacao,media_remuneracao
Advocacia-Geral da União,24560.69


**Pergunta 8:** Qual o cargo com a melhor remuneração no governo federal?

**Pergunta adicional:** Qual o quantitativo de aposentados por tipo de aposentadoria?

In [0]:
%sql
SELECT tipo_aposentadoria, COUNT(DISTINCT id) AS total_aposentados
FROM servidores.gold.fato_aposentado
GROUP BY tipo_aposentadoria
ORDER BY total_aposentados DESC

**Pergunta adicional:** Qual a distribuição de servidores por UF de exercício?

In [0]:
%sql
SELECT uf_exercicio, COUNT(DISTINCT id) AS total_servidores
FROM servidores.gold.fato_servidor
GROUP BY uf_exercicio
ORDER BY total_servidores DESC

**Pergunta adicional:** Qual a remuneração média por UF de exercício?

In [0]:
%sql
SELECT s.uf_exercicio, ROUND(AVG(r.remuneracao_apos_deducoes_obrigatorias_brl),2) AS media_remuneracao
FROM servidores.gold.fato_remuneracao r
JOIN servidores.gold.fato_servidor s ON r.id = s.id
GROUP BY s.uf_exercicio
ORDER BY media_remuneracao DESC

**Pergunta adicional:** Quais cargos possuem maior número de servidores aposentados?

In [0]:
%sql
SELECT desc_cargo, COUNT(DISTINCT id) AS total_aposentados
FROM servidores.gold.fato_aposentado
GROUP BY desc_cargo
ORDER BY total_aposentados DESC
LIMIT 5

**Pergunta adicional:** Qual a distribuição de servidores por atividade?

In [0]:
%sql
SELECT desc_atividade, COUNT(DISTINCT id) AS total
FROM servidores.gold.fato_servidor
GROUP BY desc_atividade
ORDER BY total DESC

**Pergunta adicional:** Qual a remuneração média por tipo de vínculo?

In [0]:
%sql
SELECT s.tipo_vinculo, ROUND(AVG(r.remuneracao_apos_deducoes_obrigatorias_brl),2) AS media_remuneracao
FROM servidores.gold.fato_remuneracao r
JOIN servidores.gold.fato_servidor s ON r.id = s.id
GROUP BY s.tipo_vinculo
ORDER BY media_remuneracao DESC

**Pergunta adicional:** Quantos servidores possuem mais de um vínculo ativo?

In [0]:
%sql
SELECT id, COUNT(*) AS qtd_vinculos
FROM servidores.gold.fato_servidor
GROUP BY id
HAVING COUNT(*) > 1
ORDER BY qtd_vinculos DESC

**Pergunta 9:** Quantas vagas existem em vacância no governo federal?

In [0]:
> Não é possível responder à pergunta sobre vagas ou vacância com o dataset atual, pois não há coluna de vagas/vacância nas tabelas fato ou dimensão. Seria necessário integrar dados adicionais de RH ou concursos públicos para essa análise.

**Pergunta 9:** Quantas vagas existem em vacância no governo federal?
- Não é possível responder com o dataset atual, pois não há coluna de vagas/vacância nas tabelas fato ou dimensão. Seria necessário integrar dados de RH ou concursos públicos.

**Pergunta 10:** Quais os Órgãos que possuem o maior número absoluto de vagas em vacância?
- Não há informação de vagas em vacância por órgão nas tabelas disponíveis. Exigiria fonte de dados de gestão de pessoal do governo.

**Pergunta 11:** Quais os cargos que possuem o maior número absoluto de vagas em vacância?
- Não há coluna de vagas em vacância por cargo. Seria preciso integrar dados de concursos, provimentos e vacâncias.

**Pergunta 12:** Qual a diferença na remuneração média entre servidores do sexo masculino e feminino?
- Os dados de origem para `servidores_cadastro` e `aposentado_cadastro` não possuem a coluna `sexo`. Para responder, seria necessário um atributo de gênero vinculado ao cadastro do servidor.

**Pergunta 13:** Qual a idade média ou o tempo médio de serviço dos servidores públicos federais?
- Os dados disponíveis não possuem coluna de idade ou data de nascimento, impossibilitando o cálculo dessas métricas.

**Pergunta adicional:** Qual o quantitativo de aposentados por tipo de aposentadoria?

In [0]:
%sql
SELECT tipo_aposentadoria, COUNT(DISTINCT id) AS total_aposentados
FROM servidores.gold.fato_aposentado
GROUP BY tipo_aposentadoria
ORDER BY total_aposentados DESC

**Pergunta adicional:** Qual a distribuição de servidores por UF de exercício?

In [0]:
%sql
SELECT uf_exercicio, COUNT(DISTINCT id) AS total_servidores
FROM servidores.gold.fato_servidor
GROUP BY uf_exercicio
ORDER BY total_servidores DESC

**Pergunta adicional:** Qual a remuneração média por UF de exercício?

In [0]:
%sql
SELECT s.uf_exercicio, ROUND(AVG(r.remuneracao_apos_deducoes_obrigatorias_brl),2) AS media_remuneracao
FROM servidores.gold.fato_remuneracao r
JOIN servidores.gold.fato_servidor s ON r.id = s.id
GROUP BY s.uf_exercicio
ORDER BY media_remuneracao DESC

**Pergunta adicional:** Quais cargos possuem maior número de servidores aposentados?

In [0]:
%sql
SELECT desc_cargo, COUNT(DISTINCT id) AS total_aposentados
FROM servidores.gold.fato_aposentado
GROUP BY desc_cargo
ORDER BY total_aposentados DESC
LIMIT 5

**Pergunta adicional:** Qual a distribuição de servidores por atividade?

In [0]:
%sql
SELECT desc_atividade, COUNT(DISTINCT id) AS total
FROM servidores.gold.fato_servidor
GROUP BY desc_atividade
ORDER BY total DESC

**Pergunta adicional:** Qual a remuneração média por tipo de vínculo?

In [0]:
%sql
SELECT s.tipo_vinculo, ROUND(AVG(r.remuneracao_apos_deducoes_obrigatorias_brl),2) AS media_remuneracao
FROM servidores.gold.fato_remuneracao r
JOIN servidores.gold.fato_servidor s ON r.id = s.id
GROUP BY s.tipo_vinculo
ORDER BY media_remuneracao DESC

**Pergunta adicional:** Quantos servidores possuem mais de um vínculo ativo?

In [0]:
%sql
SELECT id, COUNT(*) AS qtd_vinculos
FROM servidores.gold.fato_servidor
GROUP BY id
HAVING COUNT(*) > 1
ORDER BY qtd_vinculos DESC

## 9. Autoavaliação e Reflexão Final

Durante o desenvolvimento do pipeline, a etapa de limpeza e tratamento dos dados foi fundamental e desafiadora. Muitos atributos apresentavam baixa qualidade, como valores nulos, formatos inconsistentes de datas, códigos e descrições sem pareamento, além de registros duplicados. A padronização dos dados exigiu atenção especial, principalmente para garantir integridade referencial e confiabilidade nas análises.

A modelagem estrela também foi um ponto de maior esforço, pois exigiu decisões sobre granularidade, definição de dimensões e fatos, e ajustes para garantir flexibilidade analítica. A criação de identificadores artificiais para dimensões sem código original foi um desafio adicional.

Como sugestão para trabalhos futuros, seria interessante realizar mais agregações, criar métricas derivadas e integrar conjuntos de dados de outros órgãos, como Judiciário e Legislativo, enriquecendo ainda mais as análises e permitindo comparativos entre diferentes esferas do serviço público.

Essas etapas foram essenciais para garantir que a base GOLD esteja pronta para análises confiáveis, e a experiência contribuiu para o desenvolvimento de habilidades em engenharia de dados e modelagem dimensional.